In [ ]:
import pandas as pd
from pathlib import Path

# base path
base_path = Path("/path/to/project/results_V3/summary/MS/data/")
output_path = Path("/path/to/project/results_V3/summary/MS/")

print("=" * 60)
print("start FXC MS data( + )")
print("=" * 60)

# read mapping file( and )
print("\nreadmappingfile...")
df_peptide_map = pd.read_csv(
    base_path / "FXC.MiTRI.new_peptides_protein_map_v2.tsv",
    sep="\t"
)
print(f"PeptidemappingfilerowsSummary: {len(df_peptide_map)}")

df_species_map = pd.read_csv(
    base_path / "FXC-T-RNA_gff_protein_id_species.csv"
)
print(f"SpeciesmappingfilerowsSummary: {len(df_species_map)}")

# create peptide to protein_id mapping
peptide_to_proteins = df_peptide_map.groupby('peptide')['protein_id'].apply(
    lambda x: '|'.join(sorted(set(x.astype(str))))
).to_dict()

# species
def get_species_for_proteins(protein_ids_str):
    """
     protein_id ( | ID)find species
    """
    if pd.isna(protein_ids_str):
        return None
    protein_ids = str(protein_ids_str).split('|')
    species_list = []
    for protein_id in protein_ids:
        protein_id = protein_id.strip()
        species_match = df_species_map[df_species_map['protein_id'] == protein_id]
        if not species_match.empty:
            species_list.append(species_match.iloc[0]['species'])
            if species_list:
                return '|'.join(sorted(set(species_list)))
            return None

# ============ process data ============
print("\n" + "=" * 60)
print("process data (FXC_MS_V2.xlsx)")
print("=" * 60)

df_hla1 = pd.read_excel(base_path / "FXC_MS_V2.xlsx")
print(f" rowsSummary: {len(df_hla1)}")

# Update protein_id
df_hla1['protein_id'] = df_hla1['Peptide'].map(peptide_to_proteins)
matched_hla1 = df_hla1['protein_id'].notna().sum()
print(f"successmatched protein_id: {matched_hla1}/{len(df_hla1)}")

# species
df_hla1['species'] = df_hla1['protein_id'].apply(get_species_for_proteins)
species_matched_hla1 = df_hla1['species'].notna().sum()
print(f"successmatched species: {species_matched_hla1}/{len(df_hla1)}")

# HLA
df_hla1['HLA_type'] = 'HLAI'

# ============ process data ============
print("\n" + "=" * 60)
print("process data (FXC_MS_HLA2_V2.xlsx)")
print("=" * 60)

df_hla2 = pd.read_excel(base_path / "FXC_MS_HLA2_V2.xlsx")
print(f" rowsSummary: {len(df_hla2)}")

# Update protein_id
df_hla2['protein_id'] = df_hla2['Peptide'].map(peptide_to_proteins)
matched_hla2 = df_hla2['protein_id'].notna().sum()
print(f"successmatched protein_id: {matched_hla2}/{len(df_hla2)}")

# species
df_hla2['species'] = df_hla2['protein_id'].apply(get_species_for_proteins)
species_matched_hla2 = df_hla2['species'].notna().sum()
print(f"successmatched species: {species_matched_hla2}/{len(df_hla2)}")

# HLA
df_hla2['HLA_type'] = 'HLAII'

# ============ data ============
print("\n" + "=" * 60)
print(" and data")
print("=" * 60)

df_combined = pd.concat([df_hla1, df_hla2], ignore_index=True)
print(f" aftertotal rows: {len(df_combined)}")
print(f" - (HLAI): {len(df_hla1)} rows")
print(f" - (HLAII): {len(df_hla2)} rows")

# ============ saveresults ============
print("\n" + "=" * 60)
print("saveresults")
print("=" * 60)

output_file = output_path / "FXC_MS_V2.2.xlsx"
df_combined.to_excel(output_file, index=False)
print(f"filesavedto: {output_file}")

# ============ statistics ============
print("\n" + "=" * 60)
print(" statistics")
print("=" * 60)

print(f"\nOverall statistics:")
print(f" total rows: {len(df_combined)}")
print(f" with protein_id: {df_combined['protein_id'].notna().sum()}")
print(f" with species: {df_combined['species'].notna().sum()}")
print(f" protein_id: {df_combined['protein_id'].isna().sum()}")
print(f" species: {df_combined['species'].isna().sum()}")

print(f"\n HLA statistics:")
for hla_type in ['HLAI', 'HLAII']:
    df_type = df_combined[df_combined['HLA_type'] == hla_type]
    print(f"\n {hla_type}:")
    print(f" Summary: {len(df_type)}")
    print(f" with protein_id: {df_type['protein_id'].notna().sum()}")
    print(f" with species: {df_type['species'].notna().sum()}")

# ============ data ============
print("\n" + "=" * 60)
print("data ")
print("=" * 60)

print("\n (first3rows):")
print(df_hla1.head(3)[['Protein.Ids', 'Peptide', 'protein_id', 'species', 'HLA_type']])

print("\n (first3rows):")
print(df_hla2.head(3)[['Protein.Ids', 'Peptide', 'protein_id', 'species', 'HLA_type']])

print("\n" + "=" * 60)
print("Processing completed!")
print("=" * 60)

In [ ]:
import pandas as pd
from pathlib import Path
import re
from collections import defaultdict

# path
ms_path = Path("/path/to/project/results_V3/summary/MS/")
genes_base_path = Path("/path/to/project/results_V3/special/CRC_MS/05.MTR/05.3.MTR_genes/2.filtered_genes_CDS_version/")

print("=" * 60)
print("extractproteingene information")
print("=" * 60)

# # 1. readMSdata files
# print("\nread FXC_MS_V2.3.xlsx...")
# try:
# df_ms = pd.read_excel(ms_path / "FXC_MS_V2.3.xlsx")
# except FileNotFoundError:
# print("not found FXC_MS_V2.3.xlsx, read FXC_MS_V2.2.xlsx...")
# df_ms = pd.read_excel(ms_path / "FXC_MS_V2.2.xlsx")
df_ms = pd.read_excel(ms_path / "FXC_MS_V2.2.xlsx")
print(f"total rows: {len(df_ms)}")

# 2. build protein_id to mapping
print("\nstart species gene file...")

protein_to_coords = {} # protein_id -> coordinate information dictionary

# all species directory
species_dirs = [d for d in genes_base_path.iterdir() if d.is_dir()]
print(f"found {len(species_dirs)} speciesdirectory")

for species_dir in species_dirs:
    species_name = species_dir.name
    fna_file = species_dir / "FXC-T-RNA_target_genes_gff.fna"
    if not fna_file.exists():
        continue
    print(f" process: {species_name}")
    # fnafile
    with open(fna_file, 'r') as f:
        for line in f:
            if line.startswith('>'):
                # header rows
                # format: >NZ_CP012938.1:94984-95685 type=CDS locus_tag=Bovatus_RS00320 protein_id=cds-WP_004296942.1 gene=WP_004296942.1 product=...
                # extract (>after to first)
                parts = line.strip().split()
                coord_part = parts[0][1:] # remove'>'
                # extract
                fields = {}
                for part in parts[1:]:
                    if '=' in part:
                        key, value = part.split('=', 1)
                        fields[key] = value
                        # gene(protein_id)
                        protein_id = fields.get('gene', '')
                        if protein_id:
                            # Processing section
                            # coord_partformat: NZ_CP012938.1:94984-95685
                            if ':' in coord_part and '-' in coord_part:
                                contig, positions = coord_part.split(':')
                                start, end = positions.split('-')
                                protein_to_coords[protein_id] = {
                                    'coordinate': coord_part,
                                    'contig': contig,
                                    'start': int(start),
                                    'end': int(end),
                                    'species': species_name,
                                    'strand': fields.get('strand', ''),
                                    'locus_tag': fields.get('locus_tag', '')
                                }

print(f"\nsuccess {len(protein_to_coords)} proteins ")

# 3. for rows of data information
print("\nstartforMSdata information...")

def get_coordinates_info(protein_ids_str):
    """
     protein_id ( | ID)find information
    return:, contig, start, end
    """
    if pd.isna(protein_ids_str):
        return None, None, None, None
    protein_ids = str(protein_ids_str).split('|')
    coords_list = []
    contigs_list = []
    starts_list = []
    ends_list = []
    for protein_id in protein_ids:
        protein_id = protein_id.strip()
        if protein_id in protein_to_coords:
            coord_info = protein_to_coords[protein_id]
            coords_list.append(coord_info['coordinate'])
            contigs_list.append(coord_info['contig'])
            starts_list.append(str(coord_info['start']))
            ends_list.append(str(coord_info['end']))
            if coords_list:
                return (
                '|'.join(coords_list),
                '|'.join(contigs_list),
                '|'.join(starts_list),
                '|'.join(ends_list)
)
return None, None, None, None

# column
print(" genomic_coordinate column...")
df_ms[['genomic_coordinate', 'contig', 'gene_start', 'gene_end']] = df_ms['protein_id'].apply(
    lambda x: pd.Series(get_coordinates_info(x))
)

# statistics
coords_found = df_ms['genomic_coordinate'].notna().sum()
print(f" successfoundSummary: {coords_found}/{len(df_ms)}")

# 4. analyze speciesingene
print("\nanalyzegene gene ...")

def analyze_gene_clustering(row):
    """
    analyze speciesin genes
    """
    if pd.isna(row['genomic_coordinate']):
        return None
    coords = row['genomic_coordinate'].split('|')
    if len(coords) <= 1:
        return 'single_gene'
    # all
    gene_positions = []
    for coord in coords:
        if ':' in coord and '-' in coord:
            contig, positions = coord.split(':')
            start, end = positions.split('-')
            gene_positions.append({
                'contig': contig,
                'start': int(start),
                'end': int(end)
            })
            # check contig
            contigs = set([g['contig'] for g in gene_positions])
            if len(contigs) > 1:
                return 'multiple_contigs'
            # start
            gene_positions.sort(key=lambda x: x['start'])
            # calculategene
            max_distance = 0
            for i in range(len(gene_positions) - 1):
                distance = gene_positions[i+1]['start'] - gene_positions[i]['end']
                max_distance = max(max_distance, distance)
                # forgene cluster( <10000bpforgene cluster)
                if max_distance < 10000:
                    return f'gene_cluster(max_gap:{max_distance}bp)'
                else:
                    return f'distant(max_gap:{max_distance}bp)'

df_ms['gene_distribution'] = df_ms.apply(analyze_gene_clustering, axis=1)

# 5. saveresults
print("\nsaveresults...")
output_file = ms_path / "FXC_MS_V2.3.xlsx"
df_ms.to_excel(output_file, index=False)
print(f"savedto: {output_file}")

# 6. statistics
print("\n" + "=" * 60)
print("statistics")
print("=" * 60)

print(f"\nOverall statistics:")
print(f" total rows: {len(df_ms)}")
print(f" with information: {df_ms['genomic_coordinate'].notna().sum()}")
print(f" Summary: {df_ms['genomic_coordinate'].isna().sum()}")

print(f"\ngene statistics:")
distribution_counts = df_ms['gene_distribution'].value_counts()
for dist_type, count in distribution_counts.items():
    print(f" {dist_type}: {count}")

# 7.
print("\n" + "=" * 60)
print("data ")
print("=" * 60)

print("\n with gene cluster (first3 ):")
cluster_examples = df_ms[df_ms['gene_distribution'].str.contains('gene_cluster', na=False)].head(3)
if not cluster_examples.empty:
    print(cluster_examples[['Peptide', 'protein_id', 'species', 'genomic_coordinate', 'gene_distribution']])
else:
    print(" not found gene cluster ")

print("\n genes (first3 ):")
single_examples = df_ms[df_ms['gene_distribution'] == 'single_gene'].head(3)
if not single_examples.empty:
    print(single_examples[['Peptide', 'protein_id', 'species', 'genomic_coordinate', 'gene_distribution']])
else:
    print(" not found genes ")

print("\n" + "=" * 60)
print("Processing completed!")
print("=" * 60)

In [ ]:
import pandas as pd
from pathlib import Path
import gzip
import re
from collections import defaultdict

# path
ms_path = Path("/path/to/project/results_V3/summary/MS/")
gff_base_path = Path("/path/to/project/data/genome_annotation/gff_mic/")

print("=" * 60)
print(" GFFannotationanalyzeproteingene and ")
print("=" * 60)

# # 1. readMSdata files
# print("\nread FXC_MS_V2.3.xlsx...")
# try:
# df_ms = pd.read_excel(ms_path / "FXC_MS_V2.3.xlsx")
# except FileNotFoundError:
# print("not found FXC_MS_V2.3.xlsx, read FXC_MS_V2.2.xlsx...")
# df_ms = pd.read_excel(ms_path / "FXC_MS_V2.2.xlsx")
df_ms = pd.read_excel(ms_path / "FXC_MS_V2.2.xlsx")

print(f"total rows: {len(df_ms)}")

# 2. all process species
unique_species = set()
for species_str in df_ms['species'].dropna():
    for species in str(species_str).split('|'):
        unique_species.add(species.strip())

print(f"\n process speciesSummary: {len(unique_species)}")

# 3. buildspecies nametoGFFfilemapping
print("\nfindGFFfile...")
species_to_gff = {}
gff_files = list(gff_base_path.glob("*.gff.gz")) + list(gff_base_path.glob("*.gff"))

for gff_file in gff_files:
    filename = gff_file.name
    # extractspecies name(filenamein '_'and after '_GCF' )
    # Summary: Acidaminococcus_intestini_GCF_025144545.1_ASM2514454v1_genomic.gff.gz
    for species in unique_species:
        if species.replace('_', ' ').lower() in filename.lower().replace('_', ' ') or \
        species.lower() in filename.lower():
            species_to_gff[species] = gff_file
            break

print(f"found {len(species_to_gff)}/{len(unique_species)} speciesGFFfile")

# 4. GFFfile, buildgene
print("\n GFFfile...")

class GeneInfo:
    def __init__(self, contig, start, end, strand, protein_id, index):
        self.contig = contig
        self.start = int(start)
        self.end = int(end)
        self.strand = strand
        self.protein_id = protein_id
        self.index = index # gene order index on this contig

# species -> protein_id -> GeneInfo
species_gene_map = defaultdict(dict)

# species -> contig -> [(index, GeneInfo), ...] column gene
species_contig_genes = defaultdict(lambda: defaultdict(list))

for species, gff_file in species_to_gff.items():
    print(f" process: {species}")
    # GFFfile( )
    if gff_file.suffix == '.gz':
        f = gzip.open(gff_file, 'rt')
    else:
        f = open(gff_file, 'r')
        try:
            for line in f:
                if line.startswith('#'):
                    continue
                parts = line.strip().split('\t')
                if len(parts) < 9:
                    continue
                feature_type = parts[2]
                if feature_type != 'CDS':
                    continue
                contig = parts[0]
                start = parts[3]
                end = parts[4]
                strand = parts[6]
                attributes = parts[8]
                # extractprotein_id
                protein_id_match = re.search(r'protein_id=([^;]+)', attributes)
                if not protein_id_match:
                    continue
                protein_id = protein_id_match.group(1)
                # 'cds-'first
                if protein_id.startswith('cds-'):
                    protein_id = protein_id[4:]
                    # creategeneinformation
                    gene_info = GeneInfo(contig, start, end, strand, protein_id, 0)
                    species_gene_map[species][protein_id] = gene_info
                    species_contig_genes[species][contig].append(gene_info)
        finally:
            f.close()

# foreachcontig gene
for species in species_contig_genes:
    for contig in species_contig_genes[species]:
        # Processing section
        genes = sorted(species_contig_genes[species][contig], key=lambda x: x.start)
        # Processing section
        for idx, gene in enumerate(genes):
            gene.index = idx

print(f" {sum(len(genes) for genes in species_gene_map.values())} genes")

# 5. for rows of data and information
print("\nanalyzegene and ...")

def analyze_gene_continuity(row):
    """
    analyze peptide protein gene
    return: coordinate, contig, start, end, continuity_info
    """
    if pd.isna(row['protein_id']) or pd.isna(row['species']):
        return None, None, None, None, None
    protein_ids = str(row['protein_id']).split('|')
    species_list = str(row['species']).split('|')
    if len(protein_ids) == 1:
        # onlywith proteins, returninformation
        protein_id = protein_ids[0].strip()
        for species in species_list:
            species = species.strip()
            if species in species_gene_map and protein_id in species_gene_map[species]:
                gene = species_gene_map[species][protein_id]
                coord = f"{gene.contig}:{gene.start}-{gene.end}"
                return coord, gene.contig, gene.start, gene.end, "single_gene"
            return None, None, None, None, None
        # proteins, analyze
        coords_list = []
        gene_infos = []
        for protein_id in protein_ids:
            protein_id = protein_id.strip()
            found = False
            for species in species_list:
                species = species.strip()
                if species in species_gene_map and protein_id in species_gene_map[species]:
                    gene = species_gene_map[species][protein_id]
                    coords_list.append(f"{gene.contig}:{gene.start}-{gene.end}")
                    gene_infos.append((species, gene))
                    found = True
                    break
                if not gene_infos:
                    return None, None, None, None, None
                # check contig
                contigs = set([g[1].contig for g in gene_infos])
                if len(contigs) > 1:
                    coord_str = '|'.join(coords_list)
                    contig_str = '|'.join([g[1].contig for g in gene_infos])
                    start_str = '|'.join([str(g[1].start) for g in gene_infos])
                    end_str = '|'.join([str(g[1].end) for g in gene_infos])
                    return coord_str, contig_str, start_str, end_str, "multiple_contigs"
                # contig, calculategene
                # gene
                gene_infos_sorted = sorted(gene_infos, key=lambda x: x[1].index)
                # calculateadjacent genes genes
                gene_gaps = []
                for i in range(len(gene_infos_sorted) - 1):
                    gap = gene_infos_sorted[i+1][1].index - gene_infos_sorted[i][1].index - 1
                    gene_gaps.append(gap)
                    max_gap = max(gene_gaps) if gene_gaps else 0
                    coord_str = '|'.join(coords_list)
                    contig_str = gene_infos[0][1].contig
                    start_str = '|'.join([str(g[1].start) for g in gene_infos])
                    end_str = '|'.join([str(g[1].end) for g in gene_infos])
                    # ( in gene <=5, for gene cluster)
                    if max_gap == 0:
                        continuity = "adjacent_genes" # adjacent genes
                    elif max_gap <= 2:
                        continuity = f"close_cluster(max_gap:{max_gap}genes)" # close gene cluster
                    elif max_gap <= 5:
                        continuity = f"gene_cluster(max_gap:{max_gap}genes)" # gene cluster
                    else:
                        continuity = f"distant(max_gap:{max_gap}genes)" # distant
                        return coord_str, contig_str, start_str, end_str, continuity

# analysis
print(" gene and information...")
df_ms[['genomic_coordinate', 'contig', 'gene_start', 'gene_end', 'gene_continuity']] = \
df_ms.apply(lambda row: pd.Series(analyze_gene_continuity(row)), axis=1)

# statistics
coords_found = df_ms['genomic_coordinate'].notna().sum()
print(f" successfoundSummary: {coords_found}/{len(df_ms)}")

# 6. saveresults
print("\nsaveresults...")
output_file = ms_path / "FXC_MS_V2.3.xlsx"
df_ms.to_excel(output_file, index=False)
print(f"savedto: {output_file}")

# 7. statistics
print("\n" + "=" * 60)
print("statistics")
print("=" * 60)

print(f"\nOverall statistics:")
print(f" total rows: {len(df_ms)}")
print(f" with information: {df_ms['genomic_coordinate'].notna().sum()}")
print(f" Summary: {df_ms['genomic_coordinate'].isna().sum()}")

print(f"\ngene statistics:")
continuity_counts = df_ms['gene_continuity'].value_counts()
for cont_type, count in continuity_counts.items():
    print(f" {cont_type}: {count}")

# 8.
print("\n" + "=" * 60)
print("data ")
print("=" * 60)

print("\nadjacent genes(adjacent_genes)Summary:")
adjacent_examples = df_ms[df_ms['gene_continuity'] == 'adjacent_genes'].head(3)
if not adjacent_examples.empty:
    for idx, row in adjacent_examples.iterrows():
        print(f"\n Peptide: {row['Peptide']}")
        print(f" Protein IDs: {row['protein_id']}")
        print(f" Species: {row['species']}")
        print(f" Coordinates: {row['genomic_coordinate']}")
        print(f" Continuity: {row['gene_continuity']}")
else:
    print(" not foundadjacent genes ")

print("\ngene cluster(gene_cluster)Summary:")
cluster_examples = df_ms[df_ms['gene_continuity'].str.contains('cluster', na=False)].head(3)
if not cluster_examples.empty:
    for idx, row in cluster_examples.iterrows():
        print(f"\n Peptide: {row['Peptide']}")
        print(f" Protein IDs: {row['protein_id']}")
        print(f" Species: {row['species']}")
        print(f" Coordinates: {row['genomic_coordinate']}")
        print(f" Continuity: {row['gene_continuity']}")
else:
    print(" not found gene cluster ")

print("\n" + "=" * 60)
print("Processing completed!")
print("=" * 60)
print("\nSummary:")
print(" - adjacent_genes: gene gene (inwith coverage gene)")
print(" - close_cluster: close gene cluster(in 1-2genes)")
print(" - gene_cluster: gene cluster(in 3-5genes)")
print(" - distant: distant(in >5genes)")
print(" - multiple_contigs: /contig ")

# Processing section

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# DIA-NNresultspath
DIANN_DIR = Path("/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p")

print("=" * 80)
print("checkDIA-NNresults Prositanalyze")
print("=" * 80)

# Processing section
# 1. checkfile
# Processing section
print("\n[1]checkDIA-NNOutput files")
print("-" * 80)

main_report = DIANN_DIR / "20251212_FXC_MiTRI_HLA1p.tsv"
lib_file = DIANN_DIR / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
pg_matrix = DIANN_DIR / "20251212_FXC_MiTRI_HLA1p.pg_matrix.tsv"

files_to_check = {
    "main result file": main_report,
    "spectral library file": lib_file,
    "protein quantification matrix": pg_matrix
}

for name, file_path in files_to_check.items():
    if file_path.exists():
        size = file_path.stat().st_size / (1024*1024) # MB
        print(f"OK {name}: {file_path.name} ({size:.2f} MB)")
    else:
        print(f"Failed {name}: not found")

# Processing section
# 2. readmain result file
# Processing section
print("\n[2]readDIA-NN results")
print("-" * 80)

try:
    df = pd.read_csv(main_report, sep='\t')
    print(f"OK successfully read: {len(df)} rows of data")
    print(f"\ncolumn names ( {len(df.columns)}column):")
    for i, col in enumerate(df.columns, 1):
        print(f" {i:2d}. {col}")
except Exception as e:
    print(f"Failed readfailed: {e}")
    exit(1)

# Processing section
# 3. column check(Prosit information)
# Processing section
print("\n[3]checkProsit information")
print("-" * 80)

# Prosit information
required_info = {
    "peptide sequence": ["Stripped.Sequence", "Modified.Sequence", "Precursor.Id"],
    "precursor charge": ["Precursor.Charge"],
    "fragment information": ["Fragment.Quant.Raw", "Fragment.Quant.Corrected"],
    "mass error": ["Mass.Evidence", "CScore"],
    "identification quality": ["Q.Value", "PEP", "Global.Q.Value"]
}

found_cols = {}
for info_type, possible_cols in required_info.items():
    found = [col for col in possible_cols if col in df.columns]
    if found:
        print(f"OK {info_type}: {', '.join(found)}")
        found_cols[info_type] = found[0]
    else:
        print(f"Failed {info_type}: not found")

# Processing section
# 4. datastatistics
# Processing section
print("\n[4]datastatistics")
print("-" * 80)

if 'Stripped.Sequence' in df.columns:
    unique_peptides = df['Stripped.Sequence'].nunique()
    print(f"unique peptide count: {unique_peptides}")
    # peptide
    df['peptide_length'] = df['Stripped.Sequence'].str.len()
    length_dist = df['peptide_length'].value_counts().sort_index()
    print("\npeptideSummary:")
    for length, count in length_dist.items():
        print(f" {length:2d}mer: {count:5d} ({count/len(df)*100:.1f}%)")
        print(f"\nSummary: {df['peptide_length'].min()}-{df['peptide_length'].max()}mer")
        print(f"averageSummary: {df['peptide_length'].mean():.1f}mer")

if 'Precursor.Charge' in df.columns:
    print("\nSummary:")
    charge_dist = df['Precursor.Charge'].value_counts().sort_index()
    for charge, count in charge_dist.items():
        print(f" {charge}+: {count:5d} ({count/len(df)*100:.1f}%)")

if 'Q.Value' in df.columns:
    print("\nQ statistics:")
    thresholds = [0.01, 0.05, 0.1]
    for threshold in thresholds:
        count = (df['Q.Value'] <= threshold).sum()
        print(f" Q-value <= {threshold}: {count} ({count/len(df)*100:.1f}%)")

# Processing section
# 5. check with coverage
# Processing section
print("\n[5] after check")
print("-" * 80)

if 'Modified.Sequence' in df.columns:
    # check with coverage
    modified = df[df['Modified.Sequence'] != df['Stripped.Sequence']]
    print(f" peptide: {len(modified)} ({len(modified)/len(df)*100:.1f}%)")
    if len(modified) > 0:
        print("\n (first5):")
        for idx, row in modified.head(5).iterrows():
            print(f" Summary: {row['Stripped.Sequence']}")
            print(f" Summary: {row['Modified.Sequence']}")
            print()

# Processing section
# 6. PrositInput data
# Processing section
print("\n[6] PrositInput data")
print("-" * 80)

# filter results
if 'Q.Value' in df.columns:
    df_filtered = df[df['Q.Value'] <= 0.01].copy()
    print(f" (Q<=0.01): {len(df_filtered)}/{len(df)}")
else:
    df_filtered = df.copy()

# createPrositinputformat
if 'Stripped.Sequence' in df_filtered.columns and 'Precursor.Charge' in df_filtered.columns:
    prosit_input = df_filtered[['Stripped.Sequence', 'Precursor.Charge']].copy()
    prosit_input.columns = ['peptide_sequence', 'precursor_charge']
    # deduplicate
    prosit_input = prosit_input.drop_duplicates()
    print(f"unique peptide-Summary: {len(prosit_input)}")
    # save input file
    output_file = DIANN_DIR / "prosit_input.tsv"
    prosit_input.to_csv(output_file, sep='\t', index=False)
    print(f"\nOK Prositinput filesaved: {output_file}")
    # Processing section
    print("\nInput data (first 10):")
    print(prosit_input.head(10).to_string(index=False))

# Processing section
# 7. analysis
# Processing section
print("\n\n" + "=" * 80)
print("analysis")
print("=" * 80)

print("""
 DIA-NNresults, analysis:
[ A: Oktoberfest(, )]
---------------------------------------------
Oktoberfest forDIA-NN, completedallanalyze.

1. Summary:
pip install oktoberfest

2. rows:
oktoberfest \\
--file 20251212_FXC_MiTRI_HLA1p.tsv \\
--library prosit \\
--collision_energy 30 \\
--output ./prosit_output \\
--plot_spectra

Summary:
OK Koina
OK calculate (SA)
OK Rescoring
OK generateandmirror spectra
OK output results file

[ B: + ( )]
---------------------------------------------
 only peptide:

1. Summary:
pip install requests numpy matplotlib pandas

2. firstSummary:
- fromprosit_input.tsvreadpeptide
- Koina API
- mirror spectra

[ C: Koina API( )]
---------------------------------------------
 to in .

APISummary:
https://koina.wilhelmlab.org:443/v2/models/Prosit_2020_intensity_HCD/infer

inputformat:
{"inputs": [
{"name": "peptide_sequences", "data": ["PEPTIDE"]},
{"name": "precursor_charges", "data": [2]},
{"name": "collision_energies", "data": [30.0]}
]}

""")

# Processing section
# 8. Summary: first5peptides
# Processing section
print("\n[ ]Koinafirst5peptides")
print("-" * 80)

if 'prosit_input' in locals():
    import requests
    print(" Koina API...")
    for idx, row in prosit_input.head(5).iterrows():
        peptide = row['peptide_sequence']
        charge = int(row['precursor_charge'])
        print(f"\n{idx+1}. {peptide} ({charge}+)")
        # Koina
        url = "https://koina.wilhelmlab.org:443/v2/models/Prosit_2020_intensity_HCD/infer"
        payload = {
            "id": "test",
            "inputs": [{
            "name": "peptide_sequences",
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": [peptide]
        }, {
            "name": "precursor_charges",
            "shape": [1, 1],
            "datatype": "INT32",
            "data": [charge]
        }, {
            "name": "collision_energies",
            "shape": [1, 1],
            "datatype": "FP32",
            "data": [30.0]
        }]
        }
        try:
            response = requests.post(url, json=payload, timeout=10)
            if response.ok:
                result = response.json()
                intensities = result['outputs'][0]['data']
                max_int = max(intensities)
                print(f" OK prediction succeeded! Summary: {max_int:.4f}")
            else:
                print(f" Failed failed: {response.status_code}")
        except Exception as e:
            print(f" Failed network error: {e}")

print("\n" + "=" * 80)
print("analyzecompleted!")
print("=" * 80)

In [ ]:
"""
fromDIA-NNresultsextract experimental spectra
 sample

Summary: pip install pandas numpy matplotlib requests
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from pathlib import Path
import re
from collections import defaultdict

# Processing section
# Target: Summary: peptide
# Processing section
TARGET_PEPTIDES = [
    "KRANYSELF",
    "LYTIPNIPY",
    "RHLNATII"
]

# filepath
DIANN_DIR = Path("/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p")
MAIN_REPORT = DIANN_DIR / "20251212_FXC_MiTRI_HLA1p.tsv"
LIBRARY_FILE = DIANN_DIR / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
OUTPUT_DIR = DIANN_DIR / "mirror_plots"
OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 80)
print("DIA-NN extract ")
print("=" * 80)

# Processing section
# 1. Read data files
# Processing section
print("\n[1]readDIA-NNresultsfile")
print("-" * 80)

print(f"main result file: {MAIN_REPORT.name}")
df_main = pd.read_csv(MAIN_REPORT, sep='\t')
print(f" OK {len(df_main)} rows of data")

print(f"\nLibraryfile: {LIBRARY_FILE.name}")
df_lib = pd.read_csv(LIBRARY_FILE, sep='\t')
print(f" OK {len(df_lib)} rows of data")

# checklibraryfilecolumn
print(f"\nLibraryfile column:")
key_cols = ['PrecursorMz', 'ProductMz', 'Annotation', 'RelativeIntensity', 'ModifiedPeptide', 'PrecursorCharge', 'ProteinId']
for col in key_cols:
    if col in df_lib.columns:
        print(f" OK {col}")
    else:
        print(f" Failed {col} (not found)")

# Processing section
# 2. extracttargetpeptidedata
# Processing section
print("\n[2]extracttargetpeptide count ")
print("-" * 80)

peptide_data = {}

for peptide in TARGET_PEPTIDES:
    print(f"\npeptide: {peptide}")
    # frommain result filefind
    matches_main = df_main[df_main['Stripped.Sequence'] == peptide]
    if matches_main.empty:
        print(f" Failed main result fileinnot found")
        continue
    # statisticssampleinformation
    samples = matches_main['File.Name'].unique()
    print(f" OK found {len(matches_main)} records")
    print(f" sample count: {len(samples)}")
    # and
    mod_seq = matches_main['Modified.Sequence'].iloc[0]
    charge = int(matches_main['Precursor.Charge'].mode()[0])
    print(f" sequence: {mod_seq}")
    print(f" Summary: {charge}+")
    print(f" sample columnSummary:")
    for sample in samples:
        count = len(matches_main[matches_main['File.Name'] == sample])
        print(f" - {Path(sample).stem}: {count} ")
        # fromlibraryfileextractfragment ionsinformation
        # Libraryinpeptideformat, matched
        matches_lib = df_lib[
            (df_lib['ModifiedPeptide'].str.contains(peptide, na=False)) |
            (df_lib['ModifiedPeptide'].str.replace(r'\[.*-\]', '', regex=True) == peptide)
        ]
        if matches_lib.empty:
            print(f" Warning Libraryinnot foundfragment ionsinformation")
        else:
            print(f" OK Libraryinfound {len(matches_lib)} fragment ion records")
            peptide_data[peptide] = {
                'main': matches_main,
                'library': matches_lib,
                'modified_sequence': mod_seq,
                'charge': charge,
                'samples': samples
            }

# Processing section
# 3. Helper functions
# Processing section

def extract_fragments_from_library(lib_df, charge):
    """fromlibrarydataextractfragment ions"""
    fragments = {}
    # filter record
    lib_filtered = lib_df[lib_df.get('PrecursorCharge', charge) == charge]
    if lib_filtered.empty:
        return {}, []
    # extractfragmentsm/zand
    mz_list = []
    intensity_list = []
    annotation_list = []
    for _, row in lib_filtered.iterrows():
        if 'ProductMz' in row and 'RelativeIntensity' in row:
            mz = row['ProductMz']
            intensity = row['RelativeIntensity']
            annotation = row.get('Annotation', '')
            if pd.notna(mz) and pd.notna(intensity):
                mz_list.append(float(mz))
                intensity_list.append(float(intensity))
                annotation_list.append(annotation)
                return {
                'mz': np.array(mz_list),
                'intensity': np.array(intensity_list),
                'annotations': annotation_list
            }, lib_filtered


def calculate_fragment_mz(peptide, ion_type, position, charge):
    """calculate fragmentsm/z"""
    aa_mass = {
        'A': 71.04, 'R': 156.10, 'N': 114.04, 'D': 115.03,
        'C': 103.01, 'E': 129.04, 'Q': 128.06, 'G': 57.02,
        'H': 137.06, 'I': 113.08, 'L': 113.08, 'K': 128.09,
        'M': 131.04, 'F': 147.07, 'P': 97.05, 'S': 87.03,
        'T': 101.05, 'W': 186.08, 'Y': 163.06, 'V': 99.07
    }
    mass = 0
    if ion_type == 'b':
        for i in range(position):
            mass += aa_mass.get(peptide[i], 110)
            mass += 1.0
    elif ion_type == 'y':
        for i in range(len(peptide) - position, len(peptide)):
            mass += aa_mass.get(peptide[i], 110)
            mass += 19.0
            return (mass + charge * 1.0073) / charge


def predict_with_koina(peptide, charge, ce=30):
    """Koina"""
    url = "https://koina.wilhelmlab.org:443/v2/models/Prosit_2020_intensity_HCD/infer"
    payload = {
        "id": "predict",
        "inputs": [{
        "name": "peptide_sequences",
        "shape": [1, 1],
        "datatype": "BYTES",
        "data": [peptide]
    }, {
        "name": "precursor_charges",
        "shape": [1, 1],
        "datatype": "INT32",
        "data": [charge]
    }, {
        "name": "collision_energies",
        "shape": [1, 1],
        "datatype": "FP32",
        "data": [float(ce)]
    }]
    }
    try:
        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        result = response.json()
        return np.array(result['outputs'][0]['data'])
    except Exception as e:
        print(f" Failed Koina failed: {e}")
        return None


def parse_prosit_output(peptide, intensities):
    """ Prositoutput"""
    fragments = {}
    peptide_len = len(peptide)
    idx = 0
    for pos in range(1, peptide_len):
        for ion_charge in [1, 2, 3]:
            if idx < len(intensities):
                fragments[f"b{pos}^{ion_charge}+"] = intensities[idx]
                idx += 1
                for pos in range(1, peptide_len):
                    for ion_charge in [1, 2, 3]:
                        if idx < len(intensities):
                            fragments[f"y{pos}^{ion_charge}+"] = intensities[idx]
                            idx += 1
                            return fragments


def calculate_spectral_angle(exp_mz, exp_int, pred_fragments, peptide, tolerance=0.05):
    """calculate (Spectral Angle)"""
    # build
    pred_dict = {}
    for ion_name, intensity in pred_fragments.items():
        if intensity > 0.01:
            match = re.match(r'([by])(\d+)\^(\d+)', ion_name)
            if match:
                ion_type, position, ion_charge = match.groups()
                mz = calculate_fragment_mz(peptide, ion_type, int(position), int(ion_charge))
                pred_dict[mz] = intensity
                # matched and
                matched_exp = []
                matched_pred = []
                for exp_m, exp_i in zip(exp_mz, exp_int):
                    best_match = None
                    best_diff = tolerance
                    for pred_m, pred_i in pred_dict.items():
                        diff = abs(exp_m - pred_m)
                        if diff < best_diff:
                            best_diff = diff
                            best_match = pred_i
                            if best_match is not None:
                                matched_exp.append(exp_i)
                                matched_pred.append(best_match)
                                if len(matched_exp) < 3:
                                    return 0.0
                                # calculatespectral angle
                                exp_vec = np.array(matched_exp)
                                pred_vec = np.array(matched_pred)
                                dot_product = np.dot(exp_vec, pred_vec)
                                norm_exp = np.linalg.norm(exp_vec)
                                norm_pred = np.linalg.norm(pred_vec)
                                if norm_exp == 0 or norm_pred == 0:
                                    return 0.0
                                cos_angle = dot_product / (norm_exp * norm_pred)
                                return cos_angle


def plot_mirror_spectrum(exp_mz, exp_intensity, pred_fragments, peptide, charge, sample_name, sa, output_file):
""" mirror spectra"""
fig, ax = plt.subplots(figsize=(16, 9))
# experimental spectra( )
if exp_mz is not None and len(exp_mz) > 0:
    exp_int_norm = (exp_intensity / exp_intensity.max()) * 100
    ax.bar(exp_mz, exp_int_norm, width=1.0,
        color='gray', alpha=0.6, label='Experimental', zorder=2)
    # Processing section
    for mz, intensity in zip(exp_mz, exp_int_norm):
        if intensity > 30:
            ax. (mz, intensity+2, f'{mz:.1f}',
                rotation=90, fontsize=7, ha='center', va='bottom')
            # Processing section
            pred_mz_list = []
            pred_int_list = []
            pred_labels = []
            for ion_name, intensity in pred_fragments.items():
                if intensity > 0.01:
                    match = re.match(r'([by])(\d+)\^(\d+)', ion_name)
                    if match:
                        ion_type, position, ion_charge = match.groups()
                        mz = calculate_fragment_mz(peptide, ion_type, int(position), int(ion_charge))
                        pred_mz_list.append(mz)
                        pred_int_list.append(intensity)
                        if int(ion_charge) == 1:
                            pred_labels.append(f"{ion_type}{position}")
                        else:
                            pred_labels.append(f"{ion_type}{position}^{ion_charge}+")
                            if len(pred_int_list) > 0:
                                pred_int_array = np.array(pred_int_list)
                                pred_int_norm = (pred_int_array / pred_int_array.max()) * 100
                                ax.bar(pred_mz_list, -pred_int_norm, width=1.0,
                                    color='#4A90E2', alpha=0.7, label='Prosit', zorder=2)
                                # Processing section
                                for mz, intensity, label in zip(pred_mz_list, pred_int_norm, pred_labels):
                                    if intensity > 20:
                                        ax. (mz, -intensity-2, label,
                                            rotation=90, fontsize=8, ha='center', va='top',
                                            color='#4A90E2', fontweight='bold')
                                        # Processing section
                                        ax.axhline(y=0, color='black', linewidth=1.2, zorder=1)
                                        ax.set_xlabel('m/z', fontsize=14, fontweight='bold')
                                        ax.set_ylabel('Relative Intensity (%)', fontsize=14, fontweight='bold')
                                        # Processing section
                                        title = f"{peptide}{'+'*charge}\n"
                                        title += f"Sample: {sample_name}\n"
                                        title += f"Spectral Angle: {sa:.3f}"
                                        ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
                                        ax.legend(loc='upper right', fontsize=12)
                                        ax.set_ylim(-110, 110)
                                        # Processing section
                                        ax. (0.02, 0.98, 'Experimental', transform=ax.transAxes,
                                            fontsize=12, fontweight='bold', va='top')
                                        ax. (0.02, 0.02, 'Prosit', transform=ax.transAxes,
                                            fontsize=12, fontweight='bold', va='bottom')
                                        plt.tight_layout()
                                        plt.savefig(output_file, dpi=300, bbox_inches='tight')
                                        plt.close()
                                        print(f" OK saved: {output_file.name}")


# Processing section
# 4. process: for peptidesxsamples
# Processing section
print("\n\n[3]start mirror spectra")
print("=" * 80)

total_plots = 0
success_plots = 0

for peptide, data in peptide_data.items():
    print(f"\npeptide: {peptide}")
    print("-" * 80)
    charge = data['charge']
    samples = data['samples']
    # Koina( peptidesonly )
    print(f" -> Koina ( ={charge}+)...")
    pred_intensities = predict_with_koina(peptide, charge)
    if pred_intensities is None:
        print(f" Failed failed, skip peptide")
        continue
    pred_fragments = parse_prosit_output(peptide, pred_intensities)
    print(f" OK prediction succeeded")
    # forsamples
    print(f"\n for {len(samples)} samplesSummary:")
    for sample in samples:
        sample_name = Path(sample).stem
        print(f"\n sample: {sample_name}")
        # extract sample data
        sample_data = data['main'][data['main']['File.Name'] == sample]
        if sample_data.empty:
            continue
        # fromlibraryextractfragment ions
        # process, actual Runor information matched
        lib_data = data['library']
        if lib_data.empty:
            print(f" Warning nolibrarydata, skip")
            continue
        fragments, _ = extract_fragments_from_library(lib_data, charge)
        if len(fragments.get('mz', [])) == 0:
            print(f" Warning nofragment ionsdata")
            continue
        exp_mz = fragments['mz']
        exp_intensity = fragments['intensity']
        print(f" Summary: {len(exp_mz)}")
        # calculate
        sa = calculate_spectral_angle(exp_mz, exp_intensity, pred_fragments, peptide)
        print(f" Summary: {sa:.3f}")
        # Processing section
        output_file = OUTPUT_DIR / f"{peptide}_z{charge}_{sample_name}_SA{sa:.3f}.png"
        plot_mirror_spectrum(exp_mz, exp_intensity, pred_fragments,
            peptide, charge, sample_name, sa, output_file)
        total_plots += 1
        success_plots += 1

# Processing section
# 5.
# Processing section
print("\n\n" + "=" * 80)
print("completed!")
print("=" * 80)
print(f"successSummary: {success_plots}/{total_plots} ")
print(f"output directory: {OUTPUT_DIR}")

print("\ngenerated files:")
for png_file in sorted(OUTPUT_DIR.glob("*.png")):
    print(f" {png_file.name}")

print("\nSummary:")
print(" - filenameformat: peptide_ _sample _SA .png")
print(" - SA (Spectral Angle):, 1 ")
print(" - experimental spectra( )")
print(" - Prosit ( )")

In [ ]:
import pandas as pd
from pathlib import Path

LIBRARY_FILE = Path("/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p/20251212_FXC_MiTRI_HLA1p-lib.tsv")

print("=" * 80)
print("checkDIA-NN Libraryfile ")
print("=" * 80)

# readlibraryfile
print(f"\nread: {LIBRARY_FILE.name}")
df = pd.read_csv(LIBRARY_FILE, sep='\t', nrows=5) # read the first 5 rows for inspection

print(f"\nallcolumn names ( {len(df.columns)}column):")
for i, col in enumerate(df.columns, 1):
    print(f" {i:2d}. {col}")

print("\nfirst5rows of dataSummary:")
print(df.head())

# find intensity column
print("\nfind column:")
intensity_cols = [col for col in df.columns if 'intensity' in col.lower() or 'height' in col.lower() or 'abundance' in col.lower()]
if intensity_cols:
    for col in intensity_cols:
        print(f" OK {col}")
else:
    print(" Failed not found intensity column")

# findfragment ions column
print("\nfindfragment ions /annotation column:")
annotation_cols = [col for col in df.columns if 'annotation' in col.lower() or 'type' in col.lower() or 'fragment' in col.lower()]
if annotation_cols:
    for col in annotation_cols:
        print(f" OK {col}")
else:
    print(" Failed not foundannotation column")

# target peptide detailed data
print("\n\n" + "=" * 80)
print(" target peptideKRANYSELFlibrarydata:")
print("=" * 80)

df_full = pd.read_csv(LIBRARY_FILE, sep='\t')
target_data = df_full[df_full['ModifiedPeptide'].str.contains('KRANYSELF', na=False)]

if not target_data.empty:
    print(f"\nfound {len(target_data)} records")
    print("\n data(first 10 rows):")
    print(target_data.head(10).to_string())
    print("\n column dataSummary:")
    for col in target_data.columns:
        print(f" {col}: {target_data[col].dtype}")
else:
    print("not founddata")

print("\n\n" + "=" * 80)
print("Summary: ")
print("=" * 80)
print(" column names, Summary:")
print("1. fragment ions column names")
print("2. fragment ions /annotation column names")
print("3. correctcolumn names")

In [ ]:
"""
Jupyter Notebook - DIA-NN Library mirror spectraanalyze
 copyto Jupyter Notebookin, cell rows
"""

# Processing section
# Cell 1: and
# Processing section

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path
import json
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Jupyter Notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 100 # adjust display resolution
plt.rcParams['font.size'] = 10

#, Summary: # %config InlineBackend.figure_format = 'retina'

print("OK Library import succeeded!")


# Processing section
# Cell 2: Path configuration and
# Processing section

# path
DIANN_DIR = Path("/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p")
LIBRARY_FILE = DIANN_DIR / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
OUTPUT_DIR = Path("/path/to/project/results_V3/special/CRC_MS/08.MS/figures")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# peptide
PEPTIDES_OF_INTEREST = ['KRANYSELF', 'LYTIPNIPY', 'RHLNATII']

# Processing section
COLLISION_ENERGY = 28 # 28 is recommended for HLA data
MIN_Q_VALUE = 0.01 # Q-value threshold

# file does not exist
if LIBRARY_FILE.exists():
    print(f"OK Library file found: {LIBRARY_FILE.name}")
    print(f"Files: output directory: {OUTPUT_DIR}")
else:
    print(f"Failed file not found: {LIBRARY_FILE}")


# Processing section
# Cell 3: - DIANNLibraryExtractor
# Processing section

class DIANNLibraryExtractor:
    """fromDIA-NN Libraryextract data"""
    def __init__(self, library_file):
        self.library_file = library_file
        print(f"Read: read DIA-NN Library: {library_file.name}")
        self.df = pd.read_csv(library_file, sep='\t')
        print(f"OK successfully read {len(self.df):,} conversion records\n")
        def extract_peptide_spectrum(self, peptide, charge=2):
            """extract peptides data"""
            peptide_data = self.df[
                (self.df['PeptideSequence'] == peptide) & (self.df['PrecursorCharge'] == charge) &
                (self.df['decoy'] == 0)
            ].copy()
            if len(peptide_data) == 0:
                return None
            spectrum = {
                'peptide': peptide,
                'charge': charge,
                'precursor_mz': peptide_data['PrecursorMz'].iloc[0],
                'rt': peptide_data['Tr_recalibrated'].iloc[0],
                'mz': peptide_data['ProductMz'].values,
                'intensity': peptide_data['LibraryIntensity'].values,
                'fragment_type': peptide_data['FragmentType'].values,
                'fragment_charge': peptide_data['FragmentCharge'].values,
                'fragment_number': peptide_data['FragmentSeriesNumber'].values,
                'sample_file': peptide_data['FileName'].iloc[0],
                'q_value': peptide_data['QValue'].iloc[0]
            }
            return spectrum
        def get_available_samples(self, peptide):
            """ peptide samplein to"""
            peptide_data = self.df[
                (self.df['PeptideSequence'] == peptide) & (self.df['decoy'] == 0)
            ]
            if len(peptide_data) == 0:
                return []
            samples = peptide_data[['FileName', 'PrecursorCharge']].drop_duplicates()
            return samples.values.tolist()
        def get_basic_stats(self):
            """ statistics"""
            stats = {
                'total_transitions': len(self.df),
                'unique_peptides': self.df['PeptideSequence'].nunique(),
                'unique_samples': self.df['FileName'].nunique(),
                'charge_distribution': self.df.groupby('PrecursorCharge')['PeptideSequence'].nunique().to_dict(),
                'avg_q_value': self.df['QValue'].mean(),
                'median_q_value': self.df['QValue'].median()
            }
            return stats


# Processing section
# Cell 4: - KoinaPrositPredictor
# Processing section

class KoinaPrositPredictor:
    """ Koina API """
    def __init__(self):
        self.base_url = "https://koina.wilhelmlab.org/v2/models/Prosit_2020_intensity_HCD/infer"
        def predict_spectrum(self, peptide, charge, collision_energy=28):
            """ peptides """
            request_data = {
                "id": "predict_spectrum",
                "inputs": [
                {
                "name": "peptide_sequences",
                "shape": [1, 1],
                "datatype": "BYTES",
                "data": [peptide]
            },
                {
                "name": "collision_energies",
                "shape": [1, 1],
                "datatype": "FP32",
                "data": [float(collision_energy)]
            },
                {
                "name": "precursor_charges",
                "shape": [1, 1],
                "datatype": "INT32",
                "data": [int(charge)]
            }
            ]
            }
            try:
                response = requests.post(self.base_url, json=request_data, timeout=30)
                response.raise_for_status()
                result = response.json()
                intensities = None
                for output in result.get('outputs', []):
                    if output['name'] == 'intensities':
                        intensities = np.array(output['data'])
                        break
                    if intensities is None:
                        raise ValueError("not found results")
                    mz_values, annotations = self._generate_theoretical_mz(peptide, charge, intensities)
                    return {
                    'peptide': peptide,
                    'charge': charge,
                    'mz': mz_values,
                    'intensity': intensities,
                    'annotations': annotations
                }
            except Exception as e:
                print(f"Failed failed: {e}")
                return None
            def _generate_theoretical_mz(self, peptide, precursor_charge, intensities):
                """
                generatem/z and annotation
                Summary: Prositoutput174 (29 x 2 x 3 )
                 generate174m/z
                """
                aa_mass = {
                    'A': 71.03711, 'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
                    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
                    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
                    'M': 131.04049, 'F': 147.06841, 'P': 97.05276, 'S': 87.03203,
                    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841
                }
                mz_values = []
                annotations = []
                length = len(peptide)
                # Prosit format: 29 ( peptide )
                MAX_POSITIONS = 29
                for ion_type in ['b', 'y']:
                    for z in [1, 2, 3]:
                        for pos in range(1, MAX_POSITIONS + 1):
                            # peptide, for0( )
                            if pos >= length:
                                mz = 0.0
                                annotation = f"{ion_type}{pos}+{z}"
                            else:
                                if ion_type == 'b':
                                    # bSummary: fromN
                                    mass = sum(aa_mass.get(aa, 0) for aa in peptide[:pos]) + 1.007825 # +H
                                else:
                                    # ySummary: fromC
                                    mass = sum(aa_mass.get(aa, 0) for aa in peptide[-pos:]) + 19.01839 # +H2O+H
                                    mz = (mass + z * 1.007825) / z
                                    annotation = f"{ion_type}{pos}+{z}"
                                    mz_values.append(mz)
                                    annotations.append(annotation)
                                    # Processing section
                                    assert len(mz_values) == 174, f"expected 174 m/z values, got{len(mz_values)}"
                                    assert len(intensities) == 174, f"expected 174 intensity values, got{len(intensities)}"
                                    return np.array(mz_values), annotations


# Processing section
# Cell 5:
# Processing section

def plot_mirror_spectrum(experimental, predicted, save_path=None, show_plot=True):
    """ mirror spectra"""
    fig, ax = plt.subplots(figsize=(14, 7))
    # normalize
    exp_intensity = experimental['intensity'] / np.max(experimental['intensity']) * 100
    pred_intensity = predicted['intensity'] / np.max(predicted['intensity']) * 100
    # experimental spectra( )
    for mz, intensity, ftype, fnum, fcharge in zip(
        experimental['mz'], exp_intensity,
        experimental['fragment_type'],
        experimental['fragment_number'],
        experimental['fragment_charge']
):
color = 'red' if ftype == 'b' else 'blue'
ax.vlines(mz, 0, intensity, color=color, linewidth=2, alpha=0.7)
# Processing section
if intensity > 30:
    label = f"{ftype}{fnum}"
    if fcharge > 1:
        label += f"+{fcharge}"
        ax. (mz, intensity + 3, label, rotation=90, va='bottom', ha='center', fontsize=8, color=color)
        # Processing section
        # filterSummary: >5% m/z>0 (does not exist )
        pred_filtered_idx = (pred_intensity > 5) & (predicted['mz'] > 0)
        pred_mz_filtered = predicted['mz'][pred_filtered_idx]
        pred_int_filtered = pred_intensity[pred_filtered_idx]
        pred_annotations_filtered = [predicted['annotations'][i] for i in range(len(pred_filtered_idx)) if pred_filtered_idx[i]]
        ax.vlines(pred_mz_filtered, 0, -pred_int_filtered, color='gray', linewidth=1.5, alpha=0.6)
        # ( >30%)
        for mz, intensity, ann in zip(pred_mz_filtered, pred_int_filtered, pred_annotations_filtered):
            if intensity > 30: # label only predicted peaks with relative intensity > 30%
                ax. (mz, -intensity - 3, ann, rotation=90, va='top', ha='center', fontsize=7, color='gray', alpha=0.7)
                # Processing section
                peptide = experimental['peptide']
                charge = experimental['charge']
                sample_name = Path(experimental['sample_file']).stem
                title = f"{peptide} - {sample_name}\n(Charge: {charge}+, Q-value: {experimental['q_value']:.4f})"
                ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
                ax.set_xlabel('m/z', fontsize=12)
                ax.set_ylabel('Relative Intensity (%)', fontsize=12)
                ax.set_ylim(-100, 120)
                ax.axhline(y=0, color='black', linewidth=0.8)
                # Processing section
                from matplotlib.lines import Line2D
                legend_elements = [
                    Line2D([0], [0], color='red', lw=2, label='Experimental b ions'),
                    Line2D([0], [0], color='blue', lw=2, label='Experimental y ions'),
                    Line2D([0], [0], color='gray', lw=2, label='Predicted (Prosit)')
                ]
                ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
                # sampleinformation
                ax. (0.02, 0.98, f"RT: {experimental['rt']:.2f} min\nCE: {COLLISION_ENERGY} NCE", transform=ax.transAxes, va='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.grid(True, alpha=0.2, axis='y')
                plt.tight_layout()
                if save_path:
                    plt.savefig(save_path, dpi=300, bbox_inches='tight')
                    print(f"Saved: saved: {save_path.name}")
                    if show_plot:
                        plt.show()
                    else:
                        plt.close()


# Processing section
# Cell 6: - readLibrary
# Processing section

# readLibrary
extractor = DIANNLibraryExtractor(LIBRARY_FILE)

# statistics
stats = extractor.get_basic_stats()

print("=" * 70)
print("Library statistics")
print("=" * 70)
print(f"unique peptide count: {stats['unique_peptides']:,}")
print(f"Summary: {stats['total_transitions']:,}")
print(f"sample count: {stats['unique_samples']}")
print(f"\nSummary:")
for charge, count in sorted(stats['charge_distribution'].items()):
    print(f" {charge}+: {count:,} peptides")
print(f"\nQ-valuestatistics:")
print(f" average: {stats['avg_q_value']:.4f}")
print(f" medianSummary: {stats['median_q_value']:.4f}")


# Processing section
# Cell 7: check peptide
# Processing section

print("\n" + "=" * 70)
print("check peptide")
print("=" * 70)

peptide_info = {}

for peptide in PEPTIDES_OF_INTEREST:
    samples = extractor.get_available_samples(peptide)
    if not samples:
        print(f"\nFailed {peptide}: not found")
        continue
    print(f"\nOK {peptide}: found {len(samples)} samples/ ")
    peptide_info[peptide] = samples
    for i, (sample_file, charge) in enumerate(samples, 1):
        sample_name = Path(sample_file).stem
        print(f" {i}. {sample_name} (charge {charge}+)")

# savefoundpeptideinformation
print(f"\n found {len(peptide_info)} peptides")


# Processing section
# Cell 8: Koina
# Processing section

predictor = KoinaPrositPredictor()
print("OK Koinapredictor initialized successfully")


# Processing section
# Cell 9: mirror spectra - process all peptide
# Processing section

print("\n" + "=" * 70)
print("start mirror spectra")
print("=" * 70)

all_plots = []

for peptide in PEPTIDES_OF_INTEREST:
    if peptide not in peptide_info:
        print(f"\nWarning skip {peptide} (not founddata)")
        continue
    print(f"\n{'=' * 70}")
    print(f"processpeptide: {peptide}")
    print(f"{'=' * 70}")
    samples = peptide_info[peptide]
    for idx, (sample_file, charge) in enumerate(samples, 1):
        sample_name = Path(sample_file).stem
        print(f"\n[{idx}/{len(samples)}] sample: {sample_name}, Summary: {charge}+")
        # extractexperimental spectra
        print(" -- extractexperimental spectra...")
        experimental = extractor.extract_peptide_spectrum(peptide, charge)
        if experimental is None:
            print(" -- Warning skip(nodata)")
            continue
        print(f" -- found {len(experimental['mz'])} fragment ions")
        # Processing section
        print(" -- Koina...")
        predicted = predictor.predict_spectrum(peptide, charge, COLLISION_ENERGY)
        if predicted is None:
            print(" -- Warning failed")
            continue
        print(" -- completed")
        # mirror spectra
        output_file = OUTPUT_DIR / f"{peptide}_charge{charge}_{sample_name}_mirror.png"
        print(" -- mirror spectra...")
        plot_mirror_spectrum(
            experimental=experimental,
            predicted=predicted,
            save_path=output_file,
            show_plot=True # show in notebook; set to False to save only
)
all_plots.append(output_file)

print("\n" + "=" * 70)
print(f"OK completed!generated total {len(all_plots)} mirror spectra")
print("=" * 70)


# Processing section
# Cell 10: generated fileslist
# Processing section

print("\nFiles: generated files:")
for i, plot_file in enumerate(all_plots, 1):
    print(f" {i}. {plot_file.name}")


# Processing section
# Cell 11: (optional)
# Processing section

def plot_library_qc(df, output_dir):
    """ Library """
    fig = plt.figure(figsize=(15, 10))
    # 1.
    ax1 = plt.subplot(3, 3, 1)
    charge_dist = df['PrecursorCharge'].value_counts().sort_index()
    ax1.bar(charge_dist.index, charge_dist.values, color='steelblue', alpha=0.7)
    ax1.set_xlabel('Precursor Charge')
    ax1.set_ylabel('Count')
    ax1.set_title('Charge State Distribution', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    # 2. Q
    ax2 = plt.subplot(3, 3, 2)
    ax2.hist(df['QValue'], bins=50, color='coral', alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Q-Value')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Q-Value Distribution', fontweight='bold')
    ax2.set_yscale('log')
    ax2.axvline(x=0.01, color='red', linestyle='--', label='0.01 threshold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    # 3. peptide
    ax3 = plt.subplot(3, 3, 3)
    peptide_lengths = df.groupby('PeptideSequence').first()['PeptideSequence'].str.len()
    ax3.hist(peptide_lengths, bins=range(6, 26), color='seagreen', alpha=0.7, edgecolor='black')
    ax3.set_xlabel('Peptide Length')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Peptide Length Distribution', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    # 4. fragment ions
    ax4 = plt.subplot(3, 3, 4)
    fragment_counts = df['FragmentType'].value_counts()
    colors = ['red' if x == 'b' else 'blue' for x in fragment_counts.index]
    ax4.bar(fragment_counts.index, fragment_counts.values, color=colors, alpha=0.7)
    ax4.set_xlabel('Fragment Type')
    ax4.set_ylabel('Count')
    ax4.set_title('Fragment Ion Type', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    # 5. Library
    ax5 = plt.subplot(3, 3, 5)
    ax5.hist(df['LibraryIntensity'], bins=50, color='orange', alpha=0.7, edgecolor='black')
    ax5.set_xlabel('Library Intensity')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Intensity Distribution', fontweight='bold')
    ax5.set_yscale('log')
    ax5.grid(True, alpha=0.3)
    # 6. RT
    ax6 = plt.subplot(3, 3, 6)
    ax6.hist(df['Tr_recalibrated'], bins=50, color='teal', alpha=0.7, edgecolor='black')
    ax6.set_xlabel('Retention Time (min)')
    ax6.set_ylabel('Frequency')
    ax6.set_title('RT Distribution', fontweight='bold')
    ax6.grid(True, alpha=0.3)
    plt.tight_layout()
    output_file = output_dir / 'library_qc_overview.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Saved: savedQCSummary: {output_file.name}")
    plt.show()

# rowsQCanalyze
print("generate...")
plot_library_qc(extractor.df, OUTPUT_DIR)


# Processing section
# Cell 12: (optional) peptide
# Processing section

# peptide statistics
peptide_summary = extractor.df.groupby('PeptideSequence').agg({
    'PrecursorCharge': lambda x: ','.join(map(str, sorted(x.unique()))),
    'QValue': 'mean',
    'LibraryIntensity': 'mean',
    'Tr_recalibrated': 'mean',
    'FileName': 'nunique',
    'transition_name': 'count'
}).reset_index()

peptide_summary.columns = [
    'Peptide', 'Charges', 'Avg_QValue', 'Avg_Intensity', 'Avg_RT', 'N_Samples', 'N_Transitions'
]

peptide_summary['Length'] = peptide_summary['Peptide'].str.len()
peptide_summary = peptide_summary.sort_values('Avg_QValue')

# save
output_csv = OUTPUT_DIR / 'peptide_summary.csv'
peptide_summary.to_csv(output_csv, index=False)
print(f"Saved: savedpeptideSummary: {output_csv.name}")

# first 10 peptide
print("\n peptide (Q ):")
display(peptide_summary.head(10))


# Processing section
# Cell 13: (optional) peptides
# Processing section

def quick_plot_peptide(peptide, charge=2, show=True, save=True):
    """ peptidesmirror spectra"""
    print(f"process: {peptide} (charge {charge}+)")
    # extract data
    experimental = extractor.extract_peptide_spectrum(peptide, charge)
    if experimental is None:
        print(f"Failed not found {peptide} (charge {charge}+)")
        return None
    # Processing section
    predicted = predictor.predict_spectrum(peptide, charge, COLLISION_ENERGY)
    if predicted is None:
        print("Failed failed")
        return None
    # Processing section
    if save:
        sample_name = Path(experimental['sample_file']).stem
        output_file = OUTPUT_DIR / f"{peptide}_charge{charge}_{sample_name}_quick.png"
    else:
        output_file = None
        plot_mirror_spectrum(experimental, predicted, output_file, show)
        return experimental, predicted

# Summary: # exp, pred = quick_plot_peptide('KRANYSELF', charge=2)


# Processing section
# completed
# Processing section

print("\n" + "=" * 70)
print("OK All notebook code cells are ready!")
print("=" * 70)
print("\nSummary:")
print("1. rows Cell 1-9 completed analyze")
print("2. Cell 10 generated fileslist")
print("3. Cell 11 generate(optional)")
print("4. Cell 12 peptide (optional)")
print("5. Cell 13 peptides ")
print(f"\nFiles: all output saveSummary: {OUTPUT_DIR}")
print("=" * 70)

In [ ]:
#!/usr/bin/env python3
"""
DIA-NNresults Prosit v2.0
Summary: 1. correctKoina API
2. record
3. detailed error information
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Processing section
DIANN_DIR = "/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p"
PEPTIDES = ["KRANYSELF", "LYTIPNIPY", "RHLNATII"]
OUTPUT_DIR = f"/path/to/project/results_V3/special/CRC_MS/08.MS//mirror_plots"

# Koina API
KOINA_BASE_URL = "https://koina.wilhelmlab.org:443/v2/models"
# available list( )
AVAILABLE_MODELS = [
    "Prosit_2020_intensity_HCD", # recommended: HCD fragmentation (most common)
    "Prosit_2020_intensity_CID", # CID fragmentation
    "Prosit_2019_intensity", # older version
    "AlphaPeptDeep_ms2_generic", # AlphaPeptDeep model
]

# HCD
PROSIT_MODEL = AVAILABLE_MODELS[0]

# Processing section
AA_MASSES = {
    'A': 71.03711, 'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
    'M': 131.04049, 'F': 147.06841, 'P': 97.05276, 'S': 87.03203,
    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841
}

H2O = 18.01056
NH3 = 17.02655
PROTON = 1.007276

# Processing section

def test_koina_connection():
    """ Koina API """
    print("\n Koina API ...")
    try:
        # Processing section
        test_url = f"{KOINA_BASE_URL}/{PROSIT_MODEL}/infer"
        payload = {
            "id": "test",
            "inputs": [
            {
            "name": "peptide_sequences",
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": ["PEPTIDE"]
        },
            {
            "name": "precursor_charges",
            "shape": [1, 1],
            "datatype": "INT32",
            "data": [2]
        },
            {
            "name": "collision_energies",
            "shape": [1, 1],
            "datatype": "FP32",
            "data": [30.0]
        }
        ]
        }
        response = requests.post(test_url, json=payload, timeout=10)
        if response.status_code == 200:
            print(f"OK Koina APIconnection succeeded!using model: {PROSIT_MODEL}")
            return True
        else:
            print(f"Failed APIreturned error: {response.status_code}")
            print(f" Summary: {response. [:200]}")
            return False
    except Exception as e:
        print(f"Failed failed: {e}")
        return False

def calculate_fragment_mz(peptide, ion_type, ion_num, charge=1, loss=None):
    """calculate fragment ionsm/z"""
    if ion_type == 'b':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[:ion_num])
    elif ion_type == 'y':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[-ion_num:]) + H2O
    else:
        return 0
    # in
    if loss == 'H2O':
        mass -= H2O
    elif loss == 'NH3':
        mass -= NH3
        mz = (mass + charge * PROTON) / charge
        return mz

def get_prosit_prediction(peptide, charge=2, collision_energy=30, model=None):
    """ Koina API Prosit """
    if model is None:
        model = PROSIT_MODEL
        url = f"{KOINA_BASE_URL}/{model}/infer"
        payload = {
            "id": f"pred_{peptide}_{charge}",
            "inputs": [
            {
            "name": "peptide_sequences",
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": [peptide]
        },
            {
            "name": "precursor_charges",
            "shape": [1, 1],
            "datatype": "INT32",
            "data": [int(charge)]
        },
            {
            "name": "collision_energies",
            "shape": [1, 1],
            "datatype": "FP32",
            "data": [float(collision_energy)]
        }
        ]
        }
        try:
            print(f" Koina: {peptide} (z={charge}, CE={collision_energy}, model={model})")
            response = requests.post(url, json=payload, timeout=30)
            if response.status_code != 200:
                print(f" Failed API error {response.status_code}: {response. [:200]}")
                return None
            result = response.json()
            # checkreturndata
            if 'outputs' not in result or len(result['outputs']) == 0:
                print(f" Failed APIreturned data format error")
                return None
            intensities = np.array(result['outputs'][0]['data']).flatten()
            # normalizeto100
            if intensities.max() > 0:
                intensities = (intensities / intensities.max()) * 100
                # buildfragment ionsinformation
                fragments = []
                idx = 0
                pep_len = len(peptide)
                #, output format
                # format: b/y x (peptide -1) x 3 (no /H2O/NH3)
                for ion_type in ['b', 'y']:
                    for ion_num in range(1, pep_len):
                        for loss in [None, 'H2O', 'NH3']:
                            if idx < len(intensities):
                                mz = calculate_fragment_mz(peptide, ion_type, ion_num, 1, loss)
                                intensity = intensities[idx]
                                label = f"{ion_type}{ion_num}"
                                if loss:
                                    label += f"-{loss}"
                                    fragments.append({
                                        'mz': mz,
                                        'intensity': intensity,
                                        'label': label,
                                        'ion_type': ion_type,
                                        'ion_num': ion_num
                                    })
                                    idx += 1
                                    print(f" OK prediction succeeded, to {len(fragments)} fragment ions")
                                    return pd.DataFrame(fragments)
        except requests.exceptions.Timeout:
            print(f" Failed request timed out")
            return None
        except Exception as e:
            print(f" Failed error: {e}")
            return None

def plot_prosit_spectrum(peptide, charge, pred_df, output_file, sample_name="", peptide_info=None):
""" Prosit """
fig, ax = plt.subplots(figsize=(14, 8))
if pred_df is None or len(pred_df) == 0:
    ax. (0.5, 0.5, 'Prediction failed', ha='center', va='center', fontsize=16, color='red')
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()
    return
    # filter ( > 1%)
    pred_filtered = pred_df[pred_df['intensity'] > 1].copy()
    if len(pred_filtered) == 0:
        ax. (0.5, 0.5, 'No peaks above 1% intensity', ha='center', va='center', fontsize=16)
        plt.tight_layout()
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()
        return
    # b andy
    b_ions = pred_filtered[pred_filtered['ion_type'] == 'b']
    y_ions = pred_filtered[pred_filtered['ion_type'] == 'y']
    # Processing section
    if len(b_ions) > 0:
        ax.bar(b_ions['mz'], b_ions['intensity'], width=0.8, color='#d62728', alpha=0.7, label='b ions', edgecolor='darkred', linewidth=0.5)
        if len(y_ions) > 0:
            ax.bar(y_ions['mz'], y_ions['intensity'], width=0.8,
                color='#1f77b4', alpha=0.7, label='y ions', edgecolor='darkblue', linewidth=0.5)
            # (first15 )
            top_peaks = pred_filtered.nlargest(15, 'intensity')
            for _, peak in top_peaks.iterrows():
                if peak['intensity'] > 10: # label only peaks > 10%
                    color = '#d62728' if peak['ion_type'] == 'b' else '#1f77b4'
                    ax. (peak['mz'], peak['intensity'] + 2, peak['label'],
                        rotation=90, va='bottom', ha='center',
                        fontsize=9, color=color, fontweight='bold')
                    # Processing section
                    title = f"{peptide}"
                    if charge > 1:
                        title += "+" * charge
                        title += f"\nProsit Prediction ({PROSIT_MODEL})"
                        if sample_name:
                            title += f"\n{sample_name}"
                            ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
                            ax.set_xlabel('m/z', fontsize=12, fontweight='bold')
                            ax.set_ylabel('Relative Intensity (%)', fontsize=12, fontweight='bold')
                            # and
                            ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
                            ax.grid(axis='y', alpha=0.3, linestyle='--')
                            ax.set_ylim(0, 110)
                            # peptide information
                            if peptide_info:
                                info_lines = []
                                if 'intensity' in peptide_info:
                                    info_lines.append(f"Intensity: {peptide_info['intensity']:.2e}")
                                    if 'rt' in peptide_info:
                                        info_lines.append(f"RT: {peptide_info['rt']:.2f} min")
                                        if 'mz' in peptide_info:
                                            info_lines.append(f"m/z: {peptide_info['mz']:.4f}")
                                            if 'n_replicates' in peptide_info:
                                                info_lines.append(f"# PSMs: {peptide_info['n_replicates']}")
                                                if info_lines:
                                                    info_ = '\n'.join(info_lines)
                                                    ax. (0.02, 0.98, info_,
                                                        transform=ax.transAxes,
                                                        fontsize=9, va='top', ha='left',
                                                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
                                                    plt.tight_layout()
                                                    plt.savefig(output_file, dpi=300, bbox_inches='tight')
                                                    print(f" OK save: {output_file.name}")
                                                    plt.close()

def select_best_psm(df_sample, intensity_col=None):
    """
    from PSMin
    Summary: record
    return: record and statistics
    """
    n_psms = len(df_sample)
    # findintensity column
    if intensity_col is None:
        for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
            if col in df_sample.columns:
                intensity_col = col
                break
            if intensity_col and intensity_col in df_sample.columns:
                # Processing section
                best_idx = df_sample[intensity_col].idxmax()
                best_row = df_sample.loc[best_idx]
                avg_intensity = df_sample[intensity_col].mean()
                std_intensity = df_sample[intensity_col].std()
            else:
                # withintensity column,
                best_row = df_sample.iloc[0]
                avg_intensity = None
                std_intensity = None
                stats = {
                    'n_psms': n_psms,
                    'avg_intensity': avg_intensity,
                    'std_intensity': std_intensity,
                    'intensity_col': intensity_col
                }
                return best_row, stats

def main():
    """ """
    print("="*70)
    print("DIA-NNresults + Prosit v2.0")
    print("="*70)
    # API
    if not test_koina_connection():
        print("\nWarning warning: Koina API failed")
        print("Summary:")
        print(" 1. ")
        print(" 2. Koina in")
        print(" 3. ")
        print("\nSummary:")
        print(" - check ")
        print(" - Summary: https://koina.wilhelmlab.org")
        response = input("\n rows-(y/n): ")
        if response.lower() != 'y':
            return
        # Create output directory
        output_path = Path(OUTPUT_DIR)
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"\noutput directory: {output_path}")
        # 1. readDIA-NN report
        print(f"\n{'='*70}")
        print(" 1: readDIA-NN results")
        print("="*70)
        diann_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p.tsv"
        if not diann_file.exists():
            print(f"Failed error: file does not exist {diann_file}")
            return
        print(f"readfile: {diann_file.name}")
        df = pd.read_csv(diann_file, sep='\t', low_memory=False)
        print(f"OK load {len(df)} rows of data")
        print(f"\nfirst5column: {', '.join(df.columns[:5])}")
        # 2. identified key columns
        print(f"\n{'='*70}")
        print(" 2: data column")
        print("="*70)
        # peptide sequencecolumn
        peptide_col = None
        for col in ['Stripped.Sequence', 'Modified.Sequence', 'Peptide']:
            if col in df.columns:
                peptide_col = col
                print(f"OK peptide column: {peptide_col}")
                break
            if not peptide_col:
                print("Failed error: not foundpeptide sequencecolumn")
                return
            # sample/filecolumn
            run_col = None
            for col in ['Run', 'File.Name', 'R.FileName']:
                if col in df.columns:
                    run_col = col
                    print(f"OK sample column: {run_col}")
                    break
                # column
                charge_col = None
                for col in ['Precursor.Charge', 'Charge']:
                    if col in df.columns:
                        charge_col = col
                        print(f"OK charge column: {charge_col}")
                        break
                    # information column
                    rt_col = 'RT' if 'RT' in df.columns else None
                    mz_col = 'Precursor.Mz' if 'Precursor.Mz' in df.columns else None
                    intensity_col = None
                    for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
                        if col in df.columns:
                            intensity_col = col
                            break
                        if rt_col:
                            print(f"OK retention time column: {rt_col}")
                            if mz_col:
                                print(f"OK m/zcolumn: {mz_col}")
                                if intensity_col:
                                    print(f"OK intensity column: {intensity_col}")
                                    # 3. filtertargetpeptide
                                    print(f"\n{'='*70}")
                                    print(" 3: filter target peptide")
                                    print("="*70)
                                    print(f"targetpeptide: {', '.join(PEPTIDES)}")
                                    df_filtered = df[df[peptide_col].isin(PEPTIDES)].copy()
                                    print(f"OK found {len(df_filtered)} matched record")
                                    if len(df_filtered) == 0:
                                        print("\nWarning warning: not foundtargetpeptide")
                                        print(f"\navailablepeptide (first 10):")
                                        print(df[peptide_col].drop_duplicates().head(10).tolist())
                                        return
                                    # found peptide statistics
                                    print("\npeptidestatistics:")
                                    for peptide in PEPTIDES:
                                        count = len(df_filtered[df_filtered[peptide_col] == peptide])
                                        if count > 0:
                                            print(f" {peptide}: {count} PSM")
                                            # 4. sample process
                                            print(f"\n{'='*70}")
                                            print(" 4: generate")
                                            print("="*70)
                                            if run_col:
                                                samples = sorted(df_filtered[run_col].unique())
                                                print(f" {len(samples)} samples")
                                            else:
                                                samples = [None]
                                                print("not found sample column, alldata forsamplesprocess")
                                                # process peptides
                                                total_plots = 0
                                                for peptide in PEPTIDES:
                                                    print(f"\n{'-'*70}")
                                                    print(f"peptide: {peptide}")
                                                    print("-"*70)
                                                    df_peptide = df_filtered[df_filtered[peptide_col] == peptide]
                                                    if len(df_peptide) == 0:
                                                        print(f" Warning skip: nodata")
                                                        continue
                                                    for sample in samples:
                                                        # filter sample data
                                                        if sample and run_col:
                                                            df_sample = df_peptide[df_peptide[run_col] == sample]
                                                            sample_short = Path(sample).stem if sample else "Unknown"
                                                        else:
                                                            df_sample = df_peptide
                                                            sample_short = "All"
                                                            if len(df_sample) == 0:
                                                                continue
                                                            # PSM( )
                                                            best_row, stats = select_best_psm(df_sample, intensity_col)
                                                            # extractinformation
                                                            charge = int(best_row[charge_col]) if charge_col else 2
                                                            peptide_info = {}
                                                            if rt_col and rt_col in best_row:
                                                                peptide_info['rt'] = best_row[rt_col]
                                                                if mz_col and mz_col in best_row:
                                                                    peptide_info['mz'] = best_row[mz_col]
                                                                    if intensity_col and intensity_col in best_row:
                                                                        peptide_info['intensity'] = best_row[intensity_col]
                                                                        peptide_info['n_replicates'] = stats['n_psms']
                                                                        print(f"\n sample: {sample_short}")
                                                                        print(f" Summary: {charge}")
                                                                        print(f" PSMscount: {stats['n_psms']}")
                                                                        if peptide_info.get('rt'):
                                                                            print(f" RT: {peptide_info['rt']:.4f} min")
                                                                            if peptide_info.get('intensity'):
                                                                                print(f" Summary: {peptide_info['intensity']:.2e}")
                                                                                if stats['avg_intensity']:
                                                                                    print(f" averageSummary: {stats['avg_intensity']:.2e} +/- {stats['std_intensity']:.2e}")
                                                                                    # Prosit
                                                                                    pred_df = get_prosit_prediction(peptide, charge=charge)
                                                                                    # generatefilename
                                                                                    output_file = output_path / f"{sample_short}_{peptide}_z{charge}.png"
                                                                                    # Processing section
                                                                                    plot_prosit_spectrum(peptide, charge, pred_df, output_file,
                                                                                        sample_name=sample_short,
                                                                                        peptide_info=peptide_info)
                                                                                    total_plots += 1
                                                                                    # Processing section
                                                                                    print(f"\n{'='*70}")
                                                                                    print(f"OK completed!generated total {total_plots} spectra")
                                                                                    print(f"output directory: {output_path}")
                                                                                    print(f"using model: {PROSIT_MODEL}")
                                                                                    print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
DIA-NNresults Prositmirror spectra v3.0
Summary: 1. Koina APIreturndata
2. fromspeclibfilereadexperimental spectra
3. mirror plot(, )
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
from pathlib import Path
import warnings
import re
warnings.filterwarnings('ignore')

# Processing section
DIANN_DIR = "/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p"
PEPTIDES = ["KRANYSELF", "LYTIPNIPY", "RHLNATII"]
OUTPUT_DIR = f"/path/to/project/results_V3/special/CRC_MS/08.MS/mirror_plots"

# Koina API
KOINA_BASE_URL = "https://koina.wilhelmlab.org:443/v2/models"
PROSIT_MODEL = "Prosit_2020_intensity_HCD"

# Processing section
AA_MASSES = {
    'A': 71.03711, 'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
    'M': 131.04049, 'F': 147.06841, 'P': 97.05276, 'S': 87.03203,
    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841
}

H2O = 18.01056
NH3 = 17.02655
PROTON = 1.007276

# Processing section

def read_diann_lib_tsv(lib_file):
    """
    readDIA-NN-lib.tsvfile, extractexperimental spectra
     files infragment ionsinformation
    """
    print(f"\nreadDIA-NN file: {lib_file.name}")
    try:
        # read file
        df_lib = pd.read_csv(lib_file, sep='\t', low_memory=False)
        print(f"OK load {len(df_lib)} rowsspectral library data")
        print(f" column names: {', '.join(df_lib.columns[:8])}...")
        spectra = {}
        # find column
        peptide_col = None
        for col in ['StrippedPeptide', 'PeptideSequence', 'ModifiedPeptide']:
            if col in df_lib.columns:
                peptide_col = col
                break
            charge_col = None
            for col in ['PrecursorCharge', 'Charge']:
                if col in df_lib.columns:
                    charge_col = col
                    break
                # fragment ions column
                fragment_cols = {
                    'mz': None,
                    'intensity': None,
                    'type': None,
                    'number': None
                }
                # findfragment ionsm/zcolumn
                for col in ['FragmentMz', 'ProductMz', 'fragment_mz']:
                    if col in df_lib.columns:
                        fragment_cols['mz'] = col
                        break
                    # findfragment ionsintensity column
                    for col in ['RelativeIntensity', 'LibraryIntensity', 'Intensity', 'fragment_intensity']:
                        if col in df_lib.columns:
                            fragment_cols['intensity'] = col
                            break
                        # find column
                        for col in ['FragmentType', 'IonType', 'fragment_type']:
                            if col in df_lib.columns:
                                fragment_cols['type'] = col
                                break
                            # find column
                            for col in ['FragmentNumber', 'IonNumber', 'fragment_number']:
                                if col in df_lib.columns:
                                    fragment_cols['number'] = col
                                    break
                                if not all([peptide_col, charge_col]):
                                    print(f"Failed not foundrequired columns (peptideorcharge column)")
                                    print(f" availablecolumn: {', '.join(df_lib.columns)}")
                                    return {}
                                if not fragment_cols['mz'] or not fragment_cols['intensity']:
                                    print(f"Failed not foundfragment ionsm/zorintensity column")
                                    print(f" availablecolumn: {', '.join(df_lib.columns)}")
                                    return {}
                                print(f"OK identified key columns:")
                                print(f" peptide: {peptide_col}")
                                print(f" Summary: {charge_col}")
                                print(f" fragmentsm/z: {fragment_cols['mz']}")
                                print(f" fragmentsSummary: {fragment_cols['intensity']}")
                                if fragment_cols['type']:
                                    print(f" Summary: {fragment_cols['type']}")
                                    # peptide and
                                    for (peptide, charge), group in df_lib.groupby([peptide_col, charge_col]):
                                        # extractfragment ions
                                        peaks = []
                                        for _, row in group.iterrows():
                                            mz = row[fragment_cols['mz']]
                                            intensity = row[fragment_cols['intensity']]
                                            if pd.notna(mz) and pd.notna(intensity):
                                                peak_data = {
                                                    'mz': float(mz),
                                                    'intensity': float(intensity)
                                                }
                                                # with coverage information, save
                                                if fragment_cols['type'] and pd.notna(row[fragment_cols['type']]):
                                                    peak_data['ion_type'] = str(row[fragment_cols['type']])
                                                    if fragment_cols['number'] and pd.notna(row[fragment_cols['number']]):
                                                        peak_data['ion_num'] = int(row[fragment_cols['number']])
                                                        peaks.append(peak_data)
                                                        if peaks:
                                                            key = f"{peptide}_z{int(charge)}"
                                                            spectra[key] = pd.DataFrame(peaks)
                                                            print(f"OK successextract {len(spectra)} peptidesexperimental spectra")
                                                            # Processing section
                                                            if len(spectra) > 0:
                                                                example_key = list(spectra.keys())[0]
                                                                print(f" Summary: {example_key} with {len(spectra[example_key])} fragment ions")
                                                                return spectra
    except Exception as e:
        print(f"Failed readfailed: {e}")
        import traceback
        traceback.print_exc()
        return {}

def calculate_fragment_mz(peptide, ion_type, ion_num, charge=1, loss=None):
    """calculate fragment ionsm/z"""
    if ion_type == 'b':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[:ion_num])
    elif ion_type == 'y':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[-ion_num:]) + H2O
    else:
        return 0
    if loss == 'H2O':
        mass -= H2O
    elif loss == 'NH3':
        mass -= NH3
        mz = (mass + charge * PROTON) / charge
        return mz

def get_prosit_prediction(peptide, charge=2, collision_energy=30):
    """ Koina API Prosit, data """
    url = f"{KOINA_BASE_URL}/{PROSIT_MODEL}/infer"
    payload = {
        "id": f"pred_{peptide}_{charge}",
        "inputs": [
        {
        "name": "peptide_sequences",
        "shape": [1, 1],
        "datatype": "BYTES",
        "data": [peptide]
    },
        {
        "name": "precursor_charges",
        "shape": [1, 1],
        "datatype": "INT32",
        "data": [int(charge)]
    },
        {
        "name": "collision_energies",
        "shape": [1, 1],
        "datatype": "FP32",
        "data": [float(collision_energy)]
    }
    ]
    }
    try:
        print(f" Koina: {peptide} (z={charge})")
        response = requests.post(url, json=payload, timeout=30)
        if response.status_code != 200:
            print(f" Failed API error {response.status_code}")
            return None
        result = response.json()
        if 'outputs' not in result or len(result['outputs']) == 0:
            print(f" Failed APIreturned data format error")
            return None
        # Summary: Koina APIreturn outputs, foundintensities
        intensities = None
        annotation = None
        for output in result['outputs']:
            output_name = output.get('name', '')
            if 'intensities' in output_name.lower() or output_name == 'out/Reshape:0':
                # intensity data
                raw_data = output['data']
                # filter data
                numeric_data = []
                for val in raw_data:
                    try:
                        numeric_data.append(float(val))
                    except (ValueError, TypeError):
                        continue
                    if numeric_data:
                        intensities = np.array(numeric_data, dtype=np.float32).flatten()
                    elif 'annotation' in output_name.lower():
                        annotation = output['data']
                        if intensities is None or len(intensities) == 0:
                            print(f" Failed not found with intensity data")
                            # information
                            print(f" APIreturnoutputs: {[o.get('name', 'unnamed') for o in result['outputs']]}")
                            return None
                        # normalizeto100
                        if intensities.max() > 0:
                            intensities = (intensities / intensities.max()) * 100
                            # buildfragment ionsinformation
                            fragments = []
                            idx = 0
                            pep_len = len(peptide)
                            for ion_type in ['b', 'y']:
                                for ion_num in range(1, pep_len):
                                    for loss in [None, 'H2O', 'NH3']:
                                        if idx < len(intensities):
                                            mz = calculate_fragment_mz(peptide, ion_type, ion_num, 1, loss)
                                            intensity = intensities[idx]
                                            label = f"{ion_type}{ion_num}"
                                            if loss:
                                                label += f"-{loss}"
                                                fragments.append({
                                                    'mz': mz,
                                                    'intensity': intensity,
                                                    'label': label,
                                                    'ion_type': ion_type,
                                                    'ion_num': ion_num
                                                })
                                                idx += 1
                                                print(f" OK prediction succeeded, to {len(fragments)} fragments")
                                                return pd.DataFrame(fragments)
    except Exception as e:
        print(f" Failed error: {e}")
        import traceback
        traceback.print_exc()
        return None

def annotate_experimental_peaks(exp_df, peptide, charge=2, mz_tolerance=0.02):
    """
    forexperimental spectra annotation
    """
    if exp_df is None or len(exp_df) == 0:
        return exp_df
    exp_df = exp_df.copy()
    exp_df['label'] = ''
    exp_df['ion_type'] = ''
    pep_len = len(peptide)
    # foreach matched
    for idx, peak in exp_df.iterrows():
        peak_mz = peak['mz']
        best_match = None
        best_error = mz_tolerance
        for ion_type in ['b', 'y']:
            for ion_num in range(1, pep_len):
                for loss in [None, 'H2O', 'NH3']:
                    theo_mz = calculate_fragment_mz(peptide, ion_type, ion_num, 1, loss)
                    error = abs(peak_mz - theo_mz)
                    if error < best_error:
                        best_error = error
                        label = f"{ion_type}{ion_num}"
                        if loss:
                            label += f"-{loss}"
                            best_match = (label, ion_type)
                            if best_match:
                                exp_df.at[idx, 'label'] = best_match[0]
                                exp_df.at[idx, 'ion_type'] = best_match[1]
                                return exp_df

def plot_mirror_spectrum(peptide, charge, exp_df, pred_df, output_file, sample_name="", peptide_info=None):
"""
 mirror spectra: experimental spectra,
"""
fig, (ax_exp, ax_pred) = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={'height_ratios': [1, 1]})
# ========== Summary: experimental spectra ==========
if exp_df is not None and len(exp_df) > 0:
    # normalize
    exp_df = exp_df.copy()
    if exp_df['intensity'].max() > 0:
        exp_df['intensity'] = (exp_df['intensity'] / exp_df['intensity'].max()) * 100
        # annotation
        exp_df = annotate_experimental_peaks(exp_df, peptide, charge)
        # b andy
        exp_b = exp_df[exp_df['ion_type'] == 'b']
        exp_y = exp_df[exp_df['ion_type'] == 'y']
        exp_unknown = exp_df[exp_df['ion_type'] == '']
        # Processing section
        if len(exp_b) > 0:
            ax_exp.bar(exp_b['mz'], exp_b['intensity'], width=0.8,
                color='#d62728', alpha=0.7, label='b ions',
                edgecolor='darkred', linewidth=0.5)
            if len(exp_y) > 0:
                ax_exp.bar(exp_y['mz'], exp_y['intensity'], width=0.8,
                    color='#1f77b4', alpha=0.7, label='y ions',
                    edgecolor='darkblue', linewidth=0.5)
                if len(exp_unknown) > 0:
                    ax_exp.bar(exp_unknown['mz'], exp_unknown['intensity'], width=0.8,
                        color='gray', alpha=0.5, label='unassigned',
                        edgecolor='darkgray', linewidth=0.5)
                    # first15
                    top_peaks = exp_df.nlargest(15, 'intensity')
                    for _, peak in top_peaks.iterrows():
                        if peak['intensity'] > 10 and peak['label']:
                            color = '#d62728' if peak['ion_type'] == 'b' else '#1f77b4'
                            ax_exp. (peak['mz'], peak['intensity'] + 2, peak['label'],
                                rotation=90, va='bottom', ha='center',
                                fontsize=8, color=color, fontweight='bold')
                            ax_exp.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                            ax_exp.legend(loc='upper right', fontsize=10)
                            ax_exp.grid(axis='y', alpha=0.3, linestyle='--')
                            ax_exp.set_ylim(0, 110)
                            ax_exp.set_title('Experimental', fontsize=12, fontweight='bold', loc='right')
                        else:
                            ax_exp. (0.5, 0.5, 'No experimental data', ha='center', va='center', fontsize=14, color='gray',
                                transform=ax_exp.transAxes)
                            ax_exp.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                            ax_exp.set_ylim(0, 110)
                            # ========== Summary: Prosit ==========
                            if pred_df is not None and len(pred_df) > 0:
                                pred_filtered = pred_df[pred_df['intensity'] > 1].copy()
                                # y ( )
                                pred_filtered['intensity'] = -pred_filtered['intensity']
                                b_ions = pred_filtered[pred_filtered['ion_type'] == 'b']
                                y_ions = pred_filtered[pred_filtered['ion_type'] == 'y']
                                if len(b_ions) > 0:
                                    ax_pred.bar(b_ions['mz'], b_ions['intensity'], width=0.8,
                                        color='#d62728', alpha=0.7, label='b ions',
                                        edgecolor='darkred', linewidth=0.5)
                                    if len(y_ions) > 0:
                                        ax_pred.bar(y_ions['mz'], y_ions['intensity'], width=0.8,
                                            color='#1f77b4', alpha=0.7, label='y ions',
                                            edgecolor='darkblue', linewidth=0.5)
                                        # Processing section
                                        top_peaks = pred_filtered.nsmallest(15, 'intensity') # because values are negative
                                        for _, peak in top_peaks.iterrows():
                                            if peak['intensity'] < -10:
                                                color = '#d62728' if peak['ion_type'] == 'b' else '#1f77b4'
                                                ax_pred. (peak['mz'], peak['intensity'] - 2, peak['label'],
                                                    rotation=90, va='top', ha='center',
                                                    fontsize=8, color=color, fontweight='bold')
                                                ax_pred.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                                                ax_pred.legend(loc='lower right', fontsize=10)
                                                ax_pred.grid(axis='y', alpha=0.3, linestyle='--')
                                                ax_pred.set_ylim(-110, 0)
                                                ax_pred.set_title('Prosit', fontsize=12, fontweight='bold', loc='right')
                                            else:
                                                ax_pred. (0.5, 0.5, 'Prediction failed',
                                                    ha='center', va='center', fontsize=14, color='red',
                                                    transform=ax_pred.transAxes)
                                                ax_pred.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                                                ax_pred.set_ylim(-110, 0)
                                                # Processing section
                                                title = f"{peptide}"
                                                if charge > 1:
                                                    title += "+" * charge
                                                    if sample_name:
                                                        title += f" - {sample_name}"
                                                        fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
                                                        # x
                                                        ax_pred.set_xlabel('m/z', fontsize=12, fontweight='bold')
                                                        # information
                                                        if peptide_info:
                                                            info_lines = []
                                                            if 'intensity' in peptide_info:
                                                                info_lines.append(f"Intensity: {peptide_info['intensity']:.2e}")
                                                                if 'rt' in peptide_info:
                                                                    info_lines.append(f"RT: {peptide_info['rt']:.2f} min")
                                                                    if 'mz' in peptide_info:
                                                                        info_lines.append(f"m/z: {peptide_info['mz']:.4f}")
                                                                        if 'n_replicates' in peptide_info:
                                                                            info_lines.append(f"# PSMs: {peptide_info['n_replicates']}")
                                                                            if info_lines:
                                                                                info_ = '\n'.join(info_lines)
                                                                                ax_exp. (0.02, 0.98, info_,
                                                                                    transform=ax_exp.transAxes,
                                                                                    fontsize=9, va='top', ha='left',
                                                                                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
                                                                                plt.tight_layout()
                                                                                plt.savefig(output_file, dpi=300, bbox_inches='tight')
                                                                                print(f" OK save: {output_file.name}")
                                                                                plt.close()

def select_best_psm(df_sample, intensity_col=None):
    """from PSMin """
    n_psms = len(df_sample)
    if intensity_col is None:
        for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
            if col in df_sample.columns:
                intensity_col = col
                break
            if intensity_col and intensity_col in df_sample.columns:
                best_idx = df_sample[intensity_col].idxmax()
                best_row = df_sample.loc[best_idx]
                avg_intensity = df_sample[intensity_col].mean()
                std_intensity = df_sample[intensity_col].std()
            else:
                best_row = df_sample.iloc[0]
                avg_intensity = None
                std_intensity = None
                stats = {
                    'n_psms': n_psms,
                    'avg_intensity': avg_intensity,
                    'std_intensity': std_intensity,
                    'intensity_col': intensity_col
                }
                return best_row, stats

def main():
    """ """
    print("="*70)
    print("DIA-NN + Prosit mirror spectra v3.0")
    print("="*70)
    # Create output directory
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"\noutput directory: {output_path}")
    # readDIA-NN file(experimental spectra)
    lib_tsv_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
    exp_spectra = {}
    if lib_tsv_file.exists():
        exp_spectra = read_diann_lib_tsv(lib_tsv_file)
    else:
        print(f"\nWarning not foundDIA-NN file: {lib_tsv_file.name}")
        print(" only Prosit ")
        print(f" Summary: find -lib.tsv file")
        # readDIA-NN report
        print(f"\n{'='*70}")
        print("readDIA-NN results")
        print("="*70)
        diann_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p.tsv"
        if not diann_file.exists():
            print(f"Failed error: file does not exist {diann_file}")
            return
        print(f"readfile: {diann_file.name}")
        df = pd.read_csv(diann_file, sep='\t', low_memory=False)
        print(f"OK load {len(df)} rows of data")
        # identified key columns
        peptide_col = None
        for col in ['Stripped.Sequence', 'Modified.Sequence', 'Peptide']:
            if col in df.columns:
                peptide_col = col
                break
            if not peptide_col:
                print("Failed error: not foundpeptide sequencecolumn")
                return
            run_col = None
            for col in ['Run', 'File.Name', 'R.FileName']:
                if col in df.columns:
                    run_col = col
                    break
                charge_col = None
                for col in ['Precursor.Charge', 'Charge']:
                    if col in df.columns:
                        charge_col = col
                        break
                    rt_col = 'RT' if 'RT' in df.columns else None
                    mz_col = 'Precursor.Mz' if 'Precursor.Mz' in df.columns else None
                    intensity_col = None
                    for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
                        if col in df.columns:
                            intensity_col = col
                            break
                        print(f"OK peptide column: {peptide_col}")
                        print(f"OK sample column: {run_col}")
                        print(f"OK charge column: {charge_col}")
                        # filtertargetpeptide
                        print(f"\n{'='*70}")
                        print("filtertargetpeptide")
                        print("="*70)
                        print(f"targetpeptide: {', '.join(PEPTIDES)}")
                        df_filtered = df[df[peptide_col].isin(PEPTIDES)].copy()
                        print(f"OK found {len(df_filtered)} matched record")
                        if len(df_filtered) == 0:
                            print("\nWarning warning: not foundtargetpeptide")
                            return
                        # generate
                        print(f"\n{'='*70}")
                        print("generatemirror spectra")
                        print("="*70)
                        if run_col:
                            samples = sorted(df_filtered[run_col].unique())
                            print(f" {len(samples)} samples")
                        else:
                            samples = [None]
                            total_plots = 0
                            for peptide in PEPTIDES:
                                print(f"\n{'-'*70}")
                                print(f"peptide: {peptide}")
                                print("-"*70)
                                df_peptide = df_filtered[df_filtered[peptide_col] == peptide]
                                if len(df_peptide) == 0:
                                    continue
                                for sample in samples:
                                    if sample and run_col:
                                        df_sample = df_peptide[df_peptide[run_col] == sample]
                                        sample_short = Path(sample).stem if sample else "Unknown"
                                    else:
                                        df_sample = df_peptide
                                        sample_short = "All"
                                        if len(df_sample) == 0:
                                            continue
                                        # PSM
                                        best_row, stats = select_best_psm(df_sample, intensity_col)
                                        charge = int(best_row[charge_col]) if charge_col else 2
                                        peptide_info = {}
                                        if rt_col and rt_col in best_row:
                                            peptide_info['rt'] = best_row[rt_col]
                                            if mz_col and mz_col in best_row:
                                                peptide_info['mz'] = best_row[mz_col]
                                                if intensity_col and intensity_col in best_row:
                                                    peptide_info['intensity'] = best_row[intensity_col]
                                                    peptide_info['n_replicates'] = stats['n_psms']
                                                    print(f"\n sample: {sample_short}")
                                                    print(f" Summary: {charge}")
                                                    print(f" PSMs: {stats['n_psms']}")
                                                    # experimental spectra
                                                    exp_key = f"{peptide}_z{charge}"
                                                    exp_df = exp_spectra.get(exp_key, None)
                                                    if exp_df is not None:
                                                        print(f" experimental spectra: {len(exp_df)} ")
                                                        # Prosit
                                                        pred_df = get_prosit_prediction(peptide, charge=charge)
                                                        # mirror spectra
                                                        # with coverage data, filename for_prosit _mirror
                                                        suffix = "_mirror" if exp_df is not None else "_prosit"
                                                        output_file = output_path / f"{sample_short}_{peptide}_z{charge}{suffix}.png"
                                                        plot_mirror_spectrum(peptide, charge, exp_df, pred_df, output_file,
                                                            sample_name=sample_short,
                                                            peptide_info=peptide_info)
                                                        total_plots += 1
                                                        print(f"\n{'='*70}")
                                                        print(f"OK completed!generated total {total_plots} mirror spectra")
                                                        print(f"output directory: {output_path}")
                                                        print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
DIA-NNresults Prositmirror spectra v3.1
Summary: 1. calculate(Spectral Angle)
2. fragment ionsmatched
3. error while processingand information
4. +2 fragment ions
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Processing section
DIANN_DIR = "/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p"
PEPTIDES = ["KRANYSELF", "LYTIPNIPY", "RHLNATII"]
OUTPUT_DIR = f"/path/to/project/results_V3/special/CRC_MS/08.MS/mirror_plots"

# Koina API
KOINA_BASE_URL = "https://koina.wilhelmlab.org:443/v2/models"
PROSIT_MODEL = "Prosit_2020_intensity_HCD"

# Processing section
AA_MASSES = {
    'A': 71.03711, 'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
    'M': 131.04049, 'F': 147.06841, 'P': 97.05276, 'S': 87.03203,
    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841
}

H2O = 18.01056
NH3 = 17.02655
PROTON = 1.007276

# ==================== new: calculate ====================

def calculate_spectral_angle(exp_df, pred_df, mz_tolerance=0.02):
    """
    calculate (Spectral Angle, SA)
    SA = cos(theta) = (A-B) / (||A|| x ||B||)
    return 0-1, 1
    """
    if exp_df is None or pred_df is None or len(exp_df) == 0 or len(pred_df) == 0:
        return None
    # normalize
    exp_df = exp_df.copy()
    pred_df = pred_df.copy()
    exp_df['intensity'] = exp_df['intensity'] / exp_df['intensity'].max() if exp_df['intensity'].max() > 0 else exp_df['intensity']
    pred_df['intensity'] = pred_df['intensity'] / pred_df['intensity'].max() if pred_df['intensity'].max() > 0 else pred_df['intensity']
    # matched and
    matched_exp = []
    matched_pred = []
    for _, pred_peak in pred_df.iterrows():
        if pred_peak['intensity'] < 0.01: # filter predicted peaks that are too weak
            continue
        # experimental spectrain
        mz_diffs = np.abs(exp_df['mz'] - pred_peak['mz'])
        min_idx = mz_diffs.idxmin()
        if mz_diffs[min_idx] <= mz_tolerance:
            matched_exp.append(exp_df.loc[min_idx, 'intensity'])
            matched_pred.append(pred_peak['intensity'])
            if len(matched_exp) < 3: # 3matched
                return None
            # calculate
            matched_exp = np.array(matched_exp)
            matched_pred = np.array(matched_pred)
            dot_product = np.dot(matched_exp, matched_pred)
            norm_exp = np.linalg.norm(matched_exp)
            norm_pred = np.linalg.norm(matched_pred)
            if norm_exp == 0 or norm_pred == 0:
                return None
            sa = dot_product / (norm_exp * norm_pred)
            return float(np.clip(sa, 0, 1)) # ensure the value stays within [0,1]

# Processing section

def read_diann_lib_tsv(lib_file):
    """readDIA-NN-lib.tsvfile, extractexperimental spectra"""
    print(f"\nreadDIA-NN file: {lib_file.name}")
    try:
        df_lib = pd.read_csv(lib_file, sep='\t', low_memory=False)
        print(f"OK load {len(df_lib)} rowsspectral library data")
        print(f" column names: {', '.join(df_lib.columns[:8])}...")
        spectra = {}
        # find column
        peptide_col = None
        for col in ['StrippedPeptide', 'PeptideSequence', 'ModifiedPeptide']:
            if col in df_lib.columns:
                peptide_col = col
                break
            charge_col = None
            for col in ['PrecursorCharge', 'Charge']:
                if col in df_lib.columns:
                    charge_col = col
                    break
                # fragment ions column
                fragment_cols = {
                    'mz': None,
                    'intensity': None,
                    'type': None,
                    'number': None
                }
                for col in ['FragmentMz', 'ProductMz', 'fragment_mz']:
                    if col in df_lib.columns:
                        fragment_cols['mz'] = col
                        break
                    for col in ['RelativeIntensity', 'LibraryIntensity', 'Intensity', 'fragment_intensity']:
                        if col in df_lib.columns:
                            fragment_cols['intensity'] = col
                            break
                        for col in ['FragmentType', 'IonType', 'fragment_type']:
                            if col in df_lib.columns:
                                fragment_cols['type'] = col
                                break
                            for col in ['FragmentNumber', 'IonNumber', 'fragment_number']:
                                if col in df_lib.columns:
                                    fragment_cols['number'] = col
                                    break
                                if not all([peptide_col, charge_col]):
                                    print(f"Failed not foundrequired columns")
                                    print(f" availablecolumn: {', '.join(df_lib.columns)}")
                                    return {}
                                if not fragment_cols['mz'] or not fragment_cols['intensity']:
                                    print(f"Failed not foundfragment ionsm/zorintensity column")
                                    return {}
                                print(f"OK identified key columns:")
                                print(f" peptide: {peptide_col}")
                                print(f" Summary: {charge_col}")
                                print(f" fragmentsm/z: {fragment_cols['mz']}")
                                print(f" fragmentsSummary: {fragment_cols['intensity']}")
                                # peptide and
                                for (peptide, charge), group in df_lib.groupby([peptide_col, charge_col]):
                                    peaks = []
                                    for _, row in group.iterrows():
                                        mz = row[fragment_cols['mz']]
                                        intensity = row[fragment_cols['intensity']]
                                        if pd.notna(mz) and pd.notna(intensity):
                                            peak_data = {
                                                'mz': float(mz),
                                                'intensity': float(intensity)
                                            }
                                            if fragment_cols['type'] and pd.notna(row[fragment_cols['type']]):
                                                peak_data['ion_type'] = str(row[fragment_cols['type']])
                                                if fragment_cols['number'] and pd.notna(row[fragment_cols['number']]):
                                                    peak_data['ion_num'] = int(row[fragment_cols['number']])
                                                    peaks.append(peak_data)
                                                    if peaks:
                                                        key = f"{peptide}_z{int(charge)}"
                                                        spectra[key] = pd.DataFrame(peaks)
                                                        print(f"OK successextract {len(spectra)} peptidesexperimental spectra")
                                                        if len(spectra) > 0:
                                                            example_key = list(spectra.keys())[0]
                                                            print(f" Summary: {example_key} with {len(spectra[example_key])} fragment ions")
                                                            return spectra
    except Exception as e:
        print(f"Failed readfailed: {e}")
        import traceback
        traceback.print_exc()
        return {}

def calculate_fragment_mz(peptide, ion_type, ion_num, charge=1, loss=None):
    """calculate fragment ionsm/z"""
    if ion_type == 'b':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[:ion_num])
    elif ion_type == 'y':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[-ion_num:]) + H2O
    else:
        return 0
    if loss == 'H2O':
        mass -= H2O
    elif loss == 'NH3':
        mass -= NH3
        mz = (mass + charge * PROTON) / charge
        return mz

def get_prosit_prediction(peptide, charge=2, collision_energy=30):
    """ Koina API Prosit """
    url = f"{KOINA_BASE_URL}/{PROSIT_MODEL}/infer"
    payload = {
        "id": f"pred_{peptide}_{charge}",
        "inputs": [
        {
        "name": "peptide_sequences",
        "shape": [1, 1],
        "datatype": "BYTES",
        "data": [peptide]
    },
        {
        "name": "precursor_charges",
        "shape": [1, 1],
        "datatype": "INT32",
        "data": [int(charge)]
    },
        {
        "name": "collision_energies",
        "shape": [1, 1],
        "datatype": "FP32",
        "data": [float(collision_energy)]
    }
    ]
    }
    try:
        print(f" Koina: {peptide} (z={charge})")
        response = requests.post(url, json=payload, timeout=30)
        if response.status_code != 200:
            print(f" Failed API error {response.status_code}")
            return None
        result = response.json()
        if 'outputs' not in result or len(result['outputs']) == 0:
            print(f" Failed APIreturned data format error")
            return None
        # findintensitiesoutput
        intensities = None
        for output in result['outputs']:
            output_name = output.get('name', '')
            if 'intensities' in output_name.lower() or output_name == 'out/Reshape:0':
                raw_data = output['data']
                numeric_data = []
                for val in raw_data:
                    try:
                        numeric_data.append(float(val))
                    except (ValueError, TypeError):
                        continue
                    if numeric_data:
                        intensities = np.array(numeric_data, dtype=np.float32).flatten()
                        break
                    if intensities is None or len(intensities) == 0:
                        print(f" Failed not found with intensity data")
                        return None
                    # normalizeto100
                    if intensities.max() > 0:
                        intensities = (intensities / intensities.max()) * 100
                        # buildfragment ionsinformation
                        fragments = []
                        idx = 0
                        pep_len = len(peptide)
                        # PrositSummary: each,, after
                        for ion_type in ['b', 'y']:
                            for ion_num in range(1, pep_len):
                                for loss in [None, 'H2O', 'NH3']:
                                    if idx < len(intensities):
                                        # only +1 fragments(Prosit 2020 )
                                        mz = calculate_fragment_mz(peptide, ion_type, ion_num, 1, loss)
                                        intensity = intensities[idx]
                                        label = f"{ion_type}{ion_num}"
                                        if loss:
                                            label += f"-{loss}"
                                            fragments.append({
                                                'mz': mz,
                                                'intensity': intensity,
                                                'label': label,
                                                'ion_type': ion_type,
                                                'ion_num': ion_num
                                            })
                                            idx += 1
                                            print(f" OK prediction succeeded, to {len(fragments)} fragments ( {idx}/{len(intensities)} )")
                                            return pd.DataFrame(fragments)
    except Exception as e:
        print(f" Failed error: {e}")
        import traceback
        traceback.print_exc()
        return None

def annotate_experimental_peaks(exp_df, peptide, charge=2, mz_tolerance=0.02):
    """forexperimental spectra annotation"""
    if exp_df is None or len(exp_df) == 0:
        return exp_df
    exp_df = exp_df.copy()
    exp_df['label'] = ''
    exp_df['ion_type'] = ''
    pep_len = len(peptide)
    # foreach matched
    for idx, peak in exp_df.iterrows():
        peak_mz = peak['mz']
        best_match = None
        best_error = mz_tolerance
        # +1and+2
        for frag_charge in [1, 2]:
            for ion_type in ['b', 'y']:
                for ion_num in range(1, pep_len):
                    for loss in [None, 'H2O', 'NH3']:
                        theo_mz = calculate_fragment_mz(peptide, ion_type, ion_num, frag_charge, loss)
                        error = abs(peak_mz - theo_mz)
                        if error < best_error:
                            best_error = error
                            label = f"{ion_type}{ion_num}"
                            if frag_charge > 1:
                                label += "+" * frag_charge
                                if loss:
                                    label += f"-{loss}"
                                    best_match = (label, ion_type)
                                    if best_match:
                                        exp_df.at[idx, 'label'] = best_match[0]
                                        exp_df.at[idx, 'ion_type'] = best_match[1]
                                        return exp_df

def plot_mirror_spectrum(peptide, charge, exp_df, pred_df, output_file, sample_name="", peptide_info=None):
""" mirror spectra, """
# calculate
sa_score = None
if exp_df is not None and pred_df is not None:
    sa_score = calculate_spectral_angle(exp_df, pred_df)
    fig, (ax_exp, ax_pred) = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={'height_ratios': [1, 1]})
    # ========== Summary: experimental spectra ==========
    if exp_df is not None and len(exp_df) > 0:
        exp_df = exp_df.copy()
        if exp_df['intensity'].max() > 0:
            exp_df['intensity'] = (exp_df['intensity'] / exp_df['intensity'].max()) * 100
            exp_df = annotate_experimental_peaks(exp_df, peptide, charge)
            exp_b = exp_df[exp_df['ion_type'] == 'b']
            exp_y = exp_df[exp_df['ion_type'] == 'y']
            exp_unknown = exp_df[exp_df['ion_type'] == '']
            if len(exp_b) > 0:
                ax_exp.bar(exp_b['mz'], exp_b['intensity'], width=0.8,
                    color='#d62728', alpha=0.7, label='b ions',
                    edgecolor='darkred', linewidth=0.5)
                if len(exp_y) > 0:
                    ax_exp.bar(exp_y['mz'], exp_y['intensity'], width=0.8,
                        color='#1f77b4', alpha=0.7, label='y ions',
                        edgecolor='darkblue', linewidth=0.5)
                    if len(exp_unknown) > 0:
                        ax_exp.bar(exp_unknown['mz'], exp_unknown['intensity'], width=0.8,
                            color='gray', alpha=0.5, label='unassigned',
                            edgecolor='darkgray', linewidth=0.5)
                        # first15
                        top_peaks = exp_df.nlargest(15, 'intensity')
                        for _, peak in top_peaks.iterrows():
                            if peak['intensity'] > 10 and peak['label']:
                                color = '#d62728' if peak['ion_type'] == 'b' else '#1f77b4'
                                ax_exp. (peak['mz'], peak['intensity'] + 2, peak['label'],
                                    rotation=90, va='bottom', ha='center',
                                    fontsize=8, color=color, fontweight='bold')
                                ax_exp.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                                ax_exp.legend(loc='upper right', fontsize=10)
                                ax_exp.grid(axis='y', alpha=0.3, linestyle='--')
                                ax_exp.set_ylim(0, 110)
                                ax_exp.set_title('Experimental', fontsize=12, fontweight='bold', loc='right')
                            else:
                                ax_exp. (0.5, 0.5, 'No experimental data', ha='center', va='center', fontsize=14, color='gray',
                                    transform=ax_exp.transAxes)
                                ax_exp.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                                ax_exp.set_ylim(0, 110)
                                # ========== Summary: Prosit ==========
                                if pred_df is not None and len(pred_df) > 0:
                                    pred_filtered = pred_df[pred_df['intensity'] > 1].copy()
                                    pred_filtered['intensity'] = -pred_filtered['intensity']
                                    b_ions = pred_filtered[pred_filtered['ion_type'] == 'b']
                                    y_ions = pred_filtered[pred_filtered['ion_type'] == 'y']
                                    if len(b_ions) > 0:
                                        ax_pred.bar(b_ions['mz'], b_ions['intensity'], width=0.8,
                                            color='#d62728', alpha=0.7, label='b ions',
                                            edgecolor='darkred', linewidth=0.5)
                                        if len(y_ions) > 0:
                                            ax_pred.bar(y_ions['mz'], y_ions['intensity'], width=0.8,
                                                color='#1f77b4', alpha=0.7, label='y ions',
                                                edgecolor='darkblue', linewidth=0.5)
                                            # Processing section
                                            top_peaks = pred_filtered.nsmallest(15, 'intensity')
                                            for _, peak in top_peaks.iterrows():
                                                if peak['intensity'] < -10:
                                                    color = '#d62728' if peak['ion_type'] == 'b' else '#1f77b4'
                                                    ax_pred. (peak['mz'], peak['intensity'] - 2, peak['label'],
                                                        rotation=90, va='top', ha='center',
                                                        fontsize=8, color=color, fontweight='bold')
                                                    ax_pred.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                                                    ax_pred.legend(loc='lower right', fontsize=10)
                                                    ax_pred.grid(axis='y', alpha=0.3, linestyle='--')
                                                    ax_pred.set_ylim(-110, 0)
                                                    ax_pred.set_title('Prosit', fontsize=12, fontweight='bold', loc='right')
                                                else:
                                                    ax_pred. (0.5, 0.5, 'Prediction failed',
                                                        ha='center', va='center', fontsize=14, color='red',
                                                        transform=ax_pred.transAxes)
                                                    ax_pred.set_ylabel('Relative Intensity (%)', fontsize=11, fontweight='bold')
                                                    ax_pred.set_ylim(-110, 0)
                                                    # Processing section
                                                    title = f"{peptide}"
                                                    if charge > 1:
                                                        title += "+" * charge
                                                        if sample_name:
                                                            title += f" - {sample_name}"
                                                            fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
                                                            # x
                                                            ax_pred.set_xlabel('m/z', fontsize=12, fontweight='bold')
                                                            # Processing section
                                                            if sa_score is not None:
                                                                ax_pred. (0.02, 0.05, f'SA={sa_score:.2f}',
                                                                    transform=ax_pred.transAxes,
                                                                    fontsize=11, fontweight='bold',
                                                                    bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
                                                                # information
                                                                if peptide_info:
                                                                    info_lines = []
                                                                    if 'intensity' in peptide_info:
                                                                        info_lines.append(f"Intensity: {peptide_info['intensity']:.2e}")
                                                                        if 'rt' in peptide_info:
                                                                            info_lines.append(f"RT: {peptide_info['rt']:.2f} min")
                                                                            if 'mz' in peptide_info:
                                                                                info_lines.append(f"m/z: {peptide_info['mz']:.4f}")
                                                                                if 'n_replicates' in peptide_info:
                                                                                    info_lines.append(f"# PSMs: {peptide_info['n_replicates']}")
                                                                                    if info_lines:
                                                                                        info_ = '\n'.join(info_lines)
                                                                                        ax_exp. (0.02, 0.98, info_,
                                                                                            transform=ax_exp.transAxes,
                                                                                            fontsize=9, va='top', ha='left',
                                                                                            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
                                                                                        plt.tight_layout()
                                                                                        plt.savefig(output_file, dpi=300, bbox_inches='tight')
                                                                                        print(f" OK save: {output_file.name}")
                                                                                        if sa_score is not None:
                                                                                            print(f" SA = {sa_score:.2f}")
                                                                                            plt.close()

def select_best_psm(df_sample, intensity_col=None):
    """from PSMin """
    n_psms = len(df_sample)
    if intensity_col is None:
        for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
            if col in df_sample.columns:
                intensity_col = col
                break
            if intensity_col and intensity_col in df_sample.columns:
                best_idx = df_sample[intensity_col].idxmax()
                best_row = df_sample.loc[best_idx]
                avg_intensity = df_sample[intensity_col].mean()
                std_intensity = df_sample[intensity_col].std()
            else:
                best_row = df_sample.iloc[0]
                avg_intensity = None
                std_intensity = None
                stats = {
                    'n_psms': n_psms,
                    'avg_intensity': avg_intensity,
                    'std_intensity': std_intensity,
                    'intensity_col': intensity_col
                }
                return best_row, stats

def main():
    """ """
    print("="*70)
    print("DIA-NN + Prosit mirror spectra v3.1")
    print("="*70)
    # Create output directory
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"\noutput directory: {output_path}")
    # readDIA-NN file
    lib_tsv_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
    exp_spectra = {}
    if lib_tsv_file.exists():
        exp_spectra = read_diann_lib_tsv(lib_tsv_file)
    else:
        print(f"\nWarning not foundDIA-NN file: {lib_tsv_file.name}")
        print(" only Prosit ")
        # readDIA-NN report
        print(f"\n{'='*70}")
        print("readDIA-NN results")
        print("="*70)
        diann_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p.tsv"
        if not diann_file.exists():
            print(f"Failed error: file does not exist {diann_file}")
            return
        print(f"readfile: {diann_file.name}")
        df = pd.read_csv(diann_file, sep='\t', low_memory=False)
        print(f"OK load {len(df)} rows of data")
        # identified key columns
        peptide_col = None
        for col in ['Stripped.Sequence', 'Modified.Sequence', 'Peptide']:
            if col in df.columns:
                peptide_col = col
                break
            if not peptide_col:
                print("Failed error: not foundpeptide sequencecolumn")
                return
            run_col = None
            for col in ['Run', 'File.Name', 'R.FileName']:
                if col in df.columns:
                    run_col = col
                    break
                charge_col = None
                for col in ['Precursor.Charge', 'Charge']:
                    if col in df.columns:
                        charge_col = col
                        break
                    rt_col = 'RT' if 'RT' in df.columns else None
                    mz_col = 'Precursor.Mz' if 'Precursor.Mz' in df.columns else None
                    intensity_col = None
                    for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
                        if col in df.columns:
                            intensity_col = col
                            break
                        print(f"OK peptide column: {peptide_col}")
                        print(f"OK sample column: {run_col}")
                        print(f"OK charge column: {charge_col}")
                        # filtertargetpeptide
                        print(f"\n{'='*70}")
                        print("filtertargetpeptide")
                        print("="*70)
                        print(f"targetpeptide: {', '.join(PEPTIDES)}")
                        df_filtered = df[df[peptide_col].isin(PEPTIDES)].copy()
                        print(f"OK found {len(df_filtered)} matched record")
                        if len(df_filtered) == 0:
                            print("\nWarning warning: not foundtargetpeptide")
                            return
                        # generate
                        print(f"\n{'='*70}")
                        print("generatemirror spectra")
                        print("="*70)
                        if run_col:
                            samples = sorted(df_filtered[run_col].unique())
                            print(f" {len(samples)} samples")
                        else:
                            samples = [None]
                            total_plots = 0
                            for peptide in PEPTIDES:
                                print(f"\n{'-'*70}")
                                print(f"peptide: {peptide}")
                                print("-"*70)
                                df_peptide = df_filtered[df_filtered[peptide_col] == peptide]
                                if len(df_peptide) == 0:
                                    continue
                                for sample in samples:
                                    if sample and run_col:
                                        df_sample = df_peptide[df_peptide[run_col] == sample]
                                        sample_short = Path(sample).stem if sample else "Unknown"
                                    else:
                                        df_sample = df_peptide
                                        sample_short = "All"
                                        if len(df_sample) == 0:
                                            continue
                                        best_row, stats = select_best_psm(df_sample, intensity_col)
                                        charge = int(best_row[charge_col]) if charge_col else 2
                                        peptide_info = {}
                                        if rt_col and rt_col in best_row:
                                            peptide_info['rt'] = best_row[rt_col]
                                            if mz_col and mz_col in best_row:
                                                peptide_info['mz'] = best_row[mz_col]
                                                if intensity_col and intensity_col in best_row:
                                                    peptide_info['intensity'] = best_row[intensity_col]
                                                    peptide_info['n_replicates'] = stats['n_psms']
                                                    print(f"\n sample: {sample_short}")
                                                    print(f" Summary: {charge}")
                                                    print(f" PSMs: {stats['n_psms']}")
                                                    # experimental spectra
                                                    exp_key = f"{peptide}_z{charge}"
                                                    exp_df = exp_spectra.get(exp_key, None)
                                                    if exp_df is not None:
                                                        print(f" experimental spectra: {len(exp_df)} ")
                                                        # Prosit
                                                        pred_df = get_prosit_prediction(peptide, charge=charge)
                                                        # mirror spectra
                                                        suffix = "_mirror" if exp_df is not None else "_prosit"
                                                        output_file = output_path / f"{sample_short}_{peptide}_z{charge}{suffix}.png"
                                                        plot_mirror_spectrum(peptide, charge, exp_df, pred_df, output_file,
                                                            sample_name=sample_short,
                                                            peptide_info=peptide_info)
                                                        total_plots += 1
                                                        print(f"\n{'='*70}")
                                                        print(f"OK completed!generated total {total_plots} mirror spectra")
                                                        print(f"output directory: {output_path}")
                                                        print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Summary: save data R
with coverage data
"""

import pandas as pd
import numpy as np
import requests
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Processing section
DIANN_DIR = "/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p"
PEPTIDES = ["KRANYSELF", "LYTIPNIPY", "RHLNATII"]
OUTPUT_DIR = f"/path/to/project/results_V3/special/CRC_MS/08.MS/mirror_plots"

# Koina API
KOINA_BASE_URL = "https://koina.wilhelmlab.org:443/v2/models"
PROSIT_MODEL = "Prosit_2020_intensity_HCD"

# Processing section
AA_MASSES = {
    'A': 71.03711, 'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
    'M': 131.04049, 'F': 147.06841, 'P': 97.05276, 'S': 87.03203,
    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841
}

H2O = 18.01056
NH3 = 17.02655
PROTON = 1.007276

# ==================== calculate ====================

def calculate_spectral_angle(exp_df, pred_df, mz_tolerance=0.02):
    """calculate """
    if exp_df is None or pred_df is None or len(exp_df) == 0 or len(pred_df) == 0:
        return None
    exp_df = exp_df.copy()
    pred_df = pred_df.copy()
    exp_df['intensity'] = exp_df['intensity'] / exp_df['intensity'].max() if exp_df['intensity'].max() > 0 else exp_df['intensity']
    pred_df['intensity'] = pred_df['intensity'] / pred_df['intensity'].max() if pred_df['intensity'].max() > 0 else pred_df['intensity']
    matched_exp = []
    matched_pred = []
    for _, pred_peak in pred_df.iterrows():
        if pred_peak['intensity'] < 0.01:
            continue
        mz_diffs = np.abs(exp_df['mz'] - pred_peak['mz'])
        min_idx = mz_diffs.idxmin()
        if mz_diffs[min_idx] <= mz_tolerance:
            matched_exp.append(exp_df.loc[min_idx, 'intensity'])
            matched_pred.append(pred_peak['intensity'])
            if len(matched_exp) < 3:
                return None
            matched_exp = np.array(matched_exp)
            matched_pred = np.array(matched_pred)
            dot_product = np.dot(matched_exp, matched_pred)
            norm_exp = np.linalg.norm(matched_exp)
            norm_pred = np.linalg.norm(matched_pred)
            if norm_exp == 0 or norm_pred == 0:
                return None
            sa = dot_product / (norm_exp * norm_pred)
            return float(np.clip(sa, 0, 1))

# Processing section

def read_diann_lib_tsv(lib_file):
    """readDIA-NN-lib.tsvfile"""
    print(f"\nreadDIA-NN file: {lib_file.name}")
    try:
        df_lib = pd.read_csv(lib_file, sep='\t', low_memory=False)
        print(f"OK load {len(df_lib)} rowsspectral library data")
        spectra = {}
        peptide_col = None
        for col in ['StrippedPeptide', 'PeptideSequence', 'ModifiedPeptide']:
            if col in df_lib.columns:
                peptide_col = col
                break
            charge_col = None
            for col in ['PrecursorCharge', 'Charge']:
                if col in df_lib.columns:
                    charge_col = col
                    break
                fragment_cols = {
                    'mz': None,
                    'intensity': None,
                    'type': None,
                    'number': None
                }
                for col in ['FragmentMz', 'ProductMz', 'fragment_mz']:
                    if col in df_lib.columns:
                        fragment_cols['mz'] = col
                        break
                    for col in ['RelativeIntensity', 'LibraryIntensity', 'Intensity', 'fragment_intensity']:
                        if col in df_lib.columns:
                            fragment_cols['intensity'] = col
                            break
                        for col in ['FragmentType', 'IonType', 'fragment_type']:
                            if col in df_lib.columns:
                                fragment_cols['type'] = col
                                break
                            for col in ['FragmentNumber', 'IonNumber', 'fragment_number']:
                                if col in df_lib.columns:
                                    fragment_cols['number'] = col
                                    break
                                if not all([peptide_col, charge_col, fragment_cols['mz'], fragment_cols['intensity']]):
                                    print(f"Failed not foundrequired columns")
                                    return {}
                                for (peptide, charge), group in df_lib.groupby([peptide_col, charge_col]):
                                    peaks = []
                                    for _, row in group.iterrows():
                                        mz = row[fragment_cols['mz']]
                                        intensity = row[fragment_cols['intensity']]
                                        if pd.notna(mz) and pd.notna(intensity):
                                            peak_data = {
                                                'mz': float(mz),
                                                'intensity': float(intensity)
                                            }
                                            if fragment_cols['type'] and pd.notna(row[fragment_cols['type']]):
                                                peak_data['ion_type'] = str(row[fragment_cols['type']])
                                                if fragment_cols['number'] and pd.notna(row[fragment_cols['number']]):
                                                    peak_data['ion_num'] = int(row[fragment_cols['number']])
                                                    peaks.append(peak_data)
                                                    if peaks:
                                                        key = f"{peptide}_z{int(charge)}"
                                                        spectra[key] = pd.DataFrame(peaks)
                                                        print(f"OK successextract {len(spectra)} peptidesexperimental spectra")
                                                        return spectra
    except Exception as e:
        print(f"Failed readfailed: {e}")
        return {}

def calculate_fragment_mz(peptide, ion_type, ion_num, charge=1, loss=None):
    """calculatefragment ionsm/z"""
    if ion_type == 'b':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[:ion_num])
    elif ion_type == 'y':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[-ion_num:]) + H2O
    else:
        return 0
    if loss == 'H2O':
        mass -= H2O
    elif loss == 'NH3':
        mass -= NH3
        mz = (mass + charge * PROTON) / charge
        return mz

def get_prosit_prediction(peptide, charge=2, collision_energy=30, include_charge2=True):
    """
     Koina API Prosit
    include_charge2: generate+2 fragments
    """
    url = f"{KOINA_BASE_URL}/{PROSIT_MODEL}/infer"
    payload = {
        "id": f"pred_{peptide}_{charge}",
        "inputs": [
        {
        "name": "peptide_sequences",
        "shape": [1, 1],
        "datatype": "BYTES",
        "data": [peptide]
    },
        {
        "name": "precursor_charges",
        "shape": [1, 1],
        "datatype": "INT32",
        "data": [int(charge)]
    },
        {
        "name": "collision_energies",
        "shape": [1, 1],
        "datatype": "FP32",
        "data": [float(collision_energy)]
    }
    ]
    }
    try:
        print(f" Koina: {peptide} (z={charge})")
        response = requests.post(url, json=payload, timeout=30)
        if response.status_code != 200:
            print(f" Failed API error {response.status_code}")
            return None
        result = response.json()
        if 'outputs' not in result or len(result['outputs']) == 0:
            print(f" Failed APIreturned data format error")
            return None
        intensities = None
        for output in result['outputs']:
            output_name = output.get('name', '')
            if 'intensities' in output_name.lower() or output_name == 'out/Reshape:0':
                raw_data = output['data']
                numeric_data = []
                for val in raw_data:
                    try:
                        numeric_data.append(float(val))
                    except (ValueError, TypeError):
                        continue
                    if numeric_data:
                        intensities = np.array(numeric_data, dtype=np.float32).flatten()
                        break
                    if intensities is None or len(intensities) == 0:
                        print(f" Failed not found with intensity data")
                        return None
                    if intensities.max() > 0:
                        intensities = (intensities / intensities.max()) * 100
                        fragments = []
                        idx = 0
                        pep_len = len(peptide)
                        # Prositreturn +1 fragments
                        for ion_type in ['b', 'y']:
                            for ion_num in range(1, pep_len):
                                for loss in [None, 'H2O', 'NH3']:
                                    if idx < len(intensities):
                                        mz = calculate_fragment_mz(peptide, ion_type, ion_num, 1, loss)
                                        intensity = intensities[idx]
                                        label = f"{ion_type}{ion_num}"
                                        if loss:
                                            label += f"-{loss}"
                                            fragments.append({
                                                'mz': mz,
                                                'intensity': intensity,
                                                'label': label,
                                                'ion_type': ion_type,
                                                'ion_num': ion_num,
                                                'charge': 1
                                            })
                                            idx += 1
                                            # optional: +2
                                            if include_charge2:
                                                for ion_type in ['b', 'y']:
                                                    for ion_num in range(1, pep_len):
                                                        # +2, +1 50% for
                                                        for loss in [None]: # neutral loss is usually not considered for +2 charge
                                                            # found +1
                                                            base_label = f"{ion_type}{ion_num}"
                                                            base_peaks = [f for f in fragments if f['label'] == base_label and f['charge'] == 1]
                                                            if base_peaks and base_peaks[0]['intensity'] > 5: # add +2 charge only for stronger peaks
                                                                mz = calculate_fragment_mz(peptide, ion_type, ion_num, 2, loss)
                                                                intensity = base_peaks[0]['intensity'] * 0.5 # estimated intensity
                                                                label = f"{ion_type}{ion_num}++"
                                                                fragments.append({
                                                                    'mz': mz,
                                                                    'intensity': intensity,
                                                                    'label': label,
                                                                    'ion_type': ion_type,
                                                                    'ion_num': ion_num,
                                                                    'charge': 2
                                                                })
                                                                print(f" OK prediction succeeded, to {len(fragments)} fragments")
                                                                return pd.DataFrame(fragments)
    except Exception as e:
        print(f" Failed error: {e}")
        return None

def annotate_experimental_peaks(exp_df, peptide, charge=2, mz_tolerance=0.02):
    """forexperimental spectra annotation"""
    if exp_df is None or len(exp_df) == 0:
        return exp_df
    exp_df = exp_df.copy()
    exp_df['label'] = ''
    exp_df['ion_type'] = ''
    exp_df['charge'] = 1
    pep_len = len(peptide)
    for idx, peak in exp_df.iterrows():
        peak_mz = peak['mz']
        best_match = None
        best_error = mz_tolerance
        for frag_charge in [1, 2]:
            for ion_type in ['b', 'y']:
                for ion_num in range(1, pep_len):
                    for loss in [None, 'H2O', 'NH3']:
                        theo_mz = calculate_fragment_mz(peptide, ion_type, ion_num, frag_charge, loss)
                        error = abs(peak_mz - theo_mz)
                        if error < best_error:
                            best_error = error
                            label = f"{ion_type}{ion_num}"
                            if frag_charge > 1:
                                label += "+" * frag_charge
                                if loss:
                                    label += f"-{loss}"
                                    best_match = (label, ion_type, frag_charge)
                                    if best_match:
                                        exp_df.at[idx, 'label'] = best_match[0]
                                        exp_df.at[idx, 'ion_type'] = best_match[1]
                                        exp_df.at[idx, 'charge'] = best_match[2]
                                        return exp_df

def save_spectrum_data(peptide, charge, exp_df, pred_df, output_file, sample_name="", peptide_info=None):
"""
save dataforCSVformat, R
"""
# calculate
sa_score = None
if exp_df is not None and pred_df is not None:
    sa_score = calculate_spectral_angle(exp_df, pred_df)
    # data
    metadata = {
        'peptide': peptide,
        'charge': charge,
        'sample': sample_name,
        'sa_score': sa_score if sa_score is not None else 'NA'
    }
    if peptide_info:
        for key in ['intensity', 'rt', 'mz', 'n_replicates']:
            if key in peptide_info:
                metadata[key] = peptide_info[key]
                # save data
                metadata_file = output_file.replace('.csv', '_metadata.json')
                with open(metadata_file, 'w') as f:
                    json.dump(metadata, f, indent=2)
                    # experimental spectradata
                    if exp_df is not None and len(exp_df) > 0:
                        exp_df_save = exp_df.copy()
                        if exp_df_save['intensity'].max() > 0:
                            exp_df_save['intensity'] = (exp_df_save['intensity'] / exp_df_save['intensity'].max()) * 100
                            exp_df_save = annotate_experimental_peaks(exp_df_save, peptide, charge)
                            exp_df_save['spectrum'] = 'experimental'
                        else:
                            exp_df_save = pd.DataFrame(columns=['mz', 'intensity', 'label', 'ion_type', 'charge', 'spectrum'])
                            # data
                            if pred_df is not None and len(pred_df) > 0:
                                pred_df_save = pred_df[pred_df['intensity'] > 1].copy()
                                pred_df_save['spectrum'] = 'predicted'
                            else:
                                pred_df_save = pd.DataFrame(columns=['mz', 'intensity', 'label', 'ion_type', 'charge', 'spectrum'])
                                # data
                                combined_df = pd.concat([exp_df_save, pred_df_save], ignore_index=True)
                                # saveforCSV
                                combined_df.to_csv(output_file, index=False)
                                print(f" OK save data: {Path(output_file).name}")
                                if sa_score is not None:
                                    print(f" SA = {sa_score:.2f}")

def select_best_psm(df_sample, intensity_col=None):
    """from PSMin """
    n_psms = len(df_sample)
    if intensity_col is None:
        for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
            if col in df_sample.columns:
                intensity_col = col
                break
            if intensity_col and intensity_col in df_sample.columns:
                best_idx = df_sample[intensity_col].idxmax()
                best_row = df_sample.loc[best_idx]
                avg_intensity = df_sample[intensity_col].mean()
                std_intensity = df_sample[intensity_col].std()
            else:
                best_row = df_sample.iloc[0]
                avg_intensity = None
                std_intensity = None
                stats = {
                    'n_psms': n_psms,
                    'avg_intensity': avg_intensity,
                    'std_intensity': std_intensity,
                    'intensity_col': intensity_col
                }
                return best_row, stats

def main():
    """ """
    print("="*70)
    print("DIA-NN data ( R )")
    print("="*70)
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"\noutput directory: {output_path}")
    # readDIA-NN
    lib_tsv_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
    exp_spectra = {}
    if lib_tsv_file.exists():
        exp_spectra = read_diann_lib_tsv(lib_tsv_file)
    else:
        print(f"\nWarning not foundDIA-NN file: {lib_tsv_file.name}")
        # readDIA-NN report
        print(f"\n{'='*70}")
        print("readDIA-NN results")
        print("="*70)
        diann_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p.tsv"
        if not diann_file.exists():
            print(f"Failed error: file does not exist {diann_file}")
            return
        print(f"readfile: {diann_file.name}")
        df = pd.read_csv(diann_file, sep='\t', low_memory=False)
        print(f"OK load {len(df)} rows of data")
        # identified key columns
        peptide_col = None
        for col in ['Stripped.Sequence', 'Modified.Sequence', 'Peptide']:
            if col in df.columns:
                peptide_col = col
                break
            if not peptide_col:
                print("Failed error: not foundpeptide sequencecolumn")
                return
            run_col = None
            for col in ['Run', 'File.Name', 'R.FileName']:
                if col in df.columns:
                    run_col = col
                    break
                charge_col = None
                for col in ['Precursor.Charge', 'Charge']:
                    if col in df.columns:
                        charge_col = col
                        break
                    rt_col = 'RT' if 'RT' in df.columns else None
                    mz_col = 'Precursor.Mz' if 'Precursor.Mz' in df.columns else None
                    intensity_col = None
                    for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
                        if col in df.columns:
                            intensity_col = col
                            break
                        # filtertargetpeptide
                        print(f"\n{'='*70}")
                        print("processtargetpeptide")
                        print("="*70)
                        df_filtered = df[df[peptide_col].isin(PEPTIDES)].copy()
                        print(f"OK found {len(df_filtered)} matched record")
                        if len(df_filtered) == 0:
                            print("\nWarning warning: not foundtargetpeptide")
                            return
                        if run_col:
                            samples = sorted(df_filtered[run_col].unique())
                            print(f" {len(samples)} samples")
                        else:
                            samples = [None]
                            total_saved = 0
                            for peptide in PEPTIDES:
                                print(f"\n{'-'*70}")
                                print(f"peptide: {peptide}")
                                print("-"*70)
                                df_peptide = df_filtered[df_filtered[peptide_col] == peptide]
                                if len(df_peptide) == 0:
                                    continue
                                for sample in samples:
                                    if sample and run_col:
                                        df_sample = df_peptide[df_peptide[run_col] == sample]
                                        sample_short = Path(sample).stem if sample else "Unknown"
                                    else:
                                        df_sample = df_peptide
                                        sample_short = "All"
                                        if len(df_sample) == 0:
                                            continue
                                        best_row, stats = select_best_psm(df_sample, intensity_col)
                                        charge = int(best_row[charge_col]) if charge_col else 2
                                        peptide_info = {}
                                        if rt_col and rt_col in best_row:
                                            peptide_info['rt'] = best_row[rt_col]
                                            if mz_col and mz_col in best_row:
                                                peptide_info['mz'] = best_row[mz_col]
                                                if intensity_col and intensity_col in best_row:
                                                    peptide_info['intensity'] = best_row[intensity_col]
                                                    peptide_info['n_replicates'] = stats['n_psms']
                                                    print(f"\n sample: {sample_short}")
                                                    print(f" Summary: {charge}")
                                                    print(f" PSMs: {stats['n_psms']}")
                                                    # experimental spectra
                                                    exp_key = f"{peptide}_z{charge}"
                                                    exp_df = exp_spectra.get(exp_key, None)
                                                    if exp_df is not None:
                                                        print(f" experimental spectra: {len(exp_df)} ")
                                                        # Prosit ( +2 )
                                                        pred_df = get_prosit_prediction(peptide, charge=charge, include_charge2=True)
                                                        # save data
                                                        output_file = output_path / f"{sample_short}_{peptide}_z{charge}_data.csv"
                                                        save_spectrum_data(peptide, charge, exp_df, pred_df, str(output_file),
                                                            sample_name=sample_short,
                                                            peptide_info=peptide_info)
                                                        total_saved += 1
                                                        print(f"\n{'='*70}")
                                                        print(f"OK completed!saved total {total_saved} data files")
                                                        print(f"output directory: {output_path}")
                                                        print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Summary: 1. AlphaPeptDeepinstrument
2. NCE
3. output
"""

import pandas as pd
import numpy as np
import requests
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Processing section
DIANN_DIR = "/path/to/project/results_V3/special/CRC_MS/08.MS/DIANN/20251212/MiTRI/HLA1p"
PEPTIDES = ["KRANYSELF", "LYTIPNIPY", "RHLNATII"]
OUTPUT_DIR = f"/path/to/project/results_V3/special/CRC_MS/08.MS/mirror_plots"

# Koina API
KOINA_BASE_URL = "https://koina.wilhelmlab.org:443/v2/models"

# Orbitrap Astral DIA + HLA peptide
AVAILABLE_MODELS = {
    # === - Astral + HLA ===
    "Prosit_2020_HCD": {
    "model_name": "Prosit_2020_intensity_HCD",
    "requires_instrument": False,
    "default_nce": 30
},
    "AlphaPeptDeep_generic": {
    "model_name": "AlphaPeptDeep_ms2_generic",
    "requires_instrument": True,
    "default_instrument": "Lumos", # Astral is similar to Lumos
    "default_nce": 30
},
    "Prosit_2019": {
    "model_name": "Prosit_2019_intensity",
    "requires_instrument": False,
    "default_nce": 30
},
}

# Processing section
MODELS_TO_TEST = ["Prosit_2020_HCD"] # recommended first choice to validate the workflow

# NCE ( )
NCE_CALIBRATION_RANGE = [20, 25, 30, 35, 40] # range recommended by the literature

# Processing section
AA_MASSES = {
    'A': 71.03711, 'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
    'M': 131.04049, 'F': 147.06841, 'P': 97.05276, 'S': 87.03203,
    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841
}

H2O = 18.01056
NH3 = 17.02655
PROTON = 1.007276

# ==================== calculate ====================

def calculate_spectral_angle(exp_df, pred_df, mz_tolerance=0.02):
    """calculate """
    if exp_df is None or pred_df is None or len(exp_df) == 0 or len(pred_df) == 0:
        return None, 0
    exp_df = exp_df.copy()
    pred_df = pred_df.copy()
    # normalize
    exp_df['intensity'] = exp_df['intensity'] / exp_df['intensity'].max() if exp_df['intensity'].max() > 0 else exp_df['intensity']
    pred_df['intensity'] = pred_df['intensity'] / pred_df['intensity'].max() if pred_df['intensity'].max() > 0 else pred_df['intensity']
    matched_exp = []
    matched_pred = []
    # matched
    for _, pred_peak in pred_df.iterrows():
        if pred_peak['intensity'] < 0.01:
            continue
        mz_diffs = np.abs(exp_df['mz'] - pred_peak['mz'])
        min_idx = mz_diffs.idxmin()
        if mz_diffs[min_idx] <= mz_tolerance:
            matched_exp.append(exp_df.loc[min_idx, 'intensity'])
            matched_pred.append(pred_peak['intensity'])
            n_matched = len(matched_exp)
            if n_matched < 3:
                return None, n_matched
            matched_exp = np.array(matched_exp)
            matched_pred = np.array(matched_pred)
            # calculate
            dot_product = np.dot(matched_exp, matched_pred)
            norm_exp = np.linalg.norm(matched_exp)
            norm_pred = np.linalg.norm(matched_pred)
            if norm_exp == 0 or norm_pred == 0:
                return None, n_matched
            sa = dot_product / (norm_exp * norm_pred)
            return float(np.clip(sa, 0, 1)), n_matched

# Processing section

def read_diann_lib_tsv(lib_file):
    """readDIA-NN-lib.tsvfile"""
    print(f"\nreadDIA-NN file: {lib_file.name}")
    try:
        df_lib = pd.read_csv(lib_file, sep='\t', low_memory=False)
        print(f"OK load {len(df_lib)} rowsspectral library data")
        spectra = {}
        # column names
        peptide_col = None
        for col in ['StrippedPeptide', 'PeptideSequence', 'ModifiedPeptide']:
            if col in df_lib.columns:
                peptide_col = col
                break
            charge_col = None
            for col in ['PrecursorCharge', 'Charge']:
                if col in df_lib.columns:
                    charge_col = col
                    break
                fragment_cols = {
                    'mz': None,
                    'intensity': None,
                    'type': None,
                    'number': None
                }
                for col in ['FragmentMz', 'ProductMz', 'fragment_mz']:
                    if col in df_lib.columns:
                        fragment_cols['mz'] = col
                        break
                    for col in ['RelativeIntensity', 'LibraryIntensity', 'Intensity', 'fragment_intensity']:
                        if col in df_lib.columns:
                            fragment_cols['intensity'] = col
                            break
                        for col in ['FragmentType', 'IonType', 'fragment_type']:
                            if col in df_lib.columns:
                                fragment_cols['type'] = col
                                break
                            for col in ['FragmentNumber', 'IonNumber', 'fragment_number']:
                                if col in df_lib.columns:
                                    fragment_cols['number'] = col
                                    break
                                if not all([peptide_col, charge_col, fragment_cols['mz'], fragment_cols['intensity']]):
                                    print(f"Failed not foundrequired columns")
                                    print(f" availablecolumn: {df_lib.columns.tolist()}")
                                    return {}
                                # extract data
                                for (peptide, charge), group in df_lib.groupby([peptide_col, charge_col]):
                                    peaks = []
                                    for _, row in group.iterrows():
                                        mz = row[fragment_cols['mz']]
                                        intensity = row[fragment_cols['intensity']]
                                        if pd.notna(mz) and pd.notna(intensity):
                                            peak_data = {
                                                'mz': float(mz),
                                                'intensity': float(intensity)
                                            }
                                            if fragment_cols['type'] and pd.notna(row[fragment_cols['type']]):
                                                peak_data['ion_type'] = str(row[fragment_cols['type']])
                                                if fragment_cols['number'] and pd.notna(row[fragment_cols['number']]):
                                                    peak_data['ion_num'] = int(row[fragment_cols['number']])
                                                    peaks.append(peak_data)
                                                    if peaks:
                                                        key = f"{peptide}_z{int(charge)}"
                                                        spectra[key] = pd.DataFrame(peaks)
                                                        if len(peaks) < 20:
                                                            print(f" Warning {key}: onlywith {len(peaks)} ( SA )")
                                                            print(f"OK successextract {len(spectra)} peptidesexperimental spectra")
                                                            return spectra
    except Exception as e:
        print(f"Failed readfailed: {e}")
        return {}

def calculate_fragment_mz(peptide, ion_type, ion_num, charge=1, loss=None):
    """calculatefragment ionsm/z"""
    if ion_type == 'b':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[:ion_num])
    elif ion_type == 'y':
        mass = sum(AA_MASSES.get(aa, 0) for aa in peptide[-ion_num:]) + H2O
    else:
        return 0
    if loss == 'H2O':
        mass -= H2O
    elif loss == 'NH3':
        mass -= NH3
        mz = (mass + charge * PROTON) / charge
        return mz

def get_prediction(peptide, charge=2, model_config=None, collision_energy=30, include_charge2=True):
    """
    ,
    """
    if model_config is None:
        return None
    model_name = model_config["model_name"]
    url = f"{KOINA_BASE_URL}/{model_name}/infer"
    # build payload
    inputs = [
        {
        "name": "peptide_sequences",
        "shape": [1, 1],
        "datatype": "BYTES",
        "data": [peptide]
    },
        {
        "name": "precursor_charges",
        "shape": [1, 1],
        "datatype": "INT32",
        "data": [int(charge)]
    },
        {
        "name": "collision_energies",
        "shape": [1, 1],
        "datatype": "FP32",
        "data": [float(collision_energy)]
    }
    ]
    # AlphaPeptDeep instrument
    if model_config.get("requires_instrument", False):
        instrument = model_config.get("default_instrument", "Lumos")
        inputs.append({
            "name": "instrument_types",
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": [instrument]
        })
        print(f" Summary: {instrument}")
        payload = {
            "id": f"pred_{peptide}_{charge}",
            "inputs": inputs
        }
        try:
            print(f" {model_name}: {peptide} (z={charge}, NCE={collision_energy})")
            response = requests.post(url, json=payload, timeout=30)
            if response.status_code != 200:
                print(f" Failed API error {response.status_code}")
                try:
                    error_detail = response.json()
                    print(f" errorSummary: {error_detail}")
                except:
                    pass
                return None
            result = response.json()
            if 'outputs' not in result or len(result['outputs']) == 0:
                print(f" Failed APIreturned data format error")
                return None
            # extractintensity data
            intensities = None
            for output in result['outputs']:
                output_name = output.get('name', '')
                if 'intensities' in output_name.lower() or output_name == 'out/Reshape:0':
                    raw_data = output['data']
                    numeric_data = []
                    for val in raw_data:
                        try:
                            numeric_data.append(float(val))
                        except (ValueError, TypeError):
                            continue
                        if numeric_data:
                            intensities = np.array(numeric_data, dtype=np.float32).flatten()
                            break
                        if intensities is None or len(intensities) == 0:
                            print(f" Failed not found with intensity data")
                            return None
                        # normalize
                        if intensities.max() > 0:
                            intensities = (intensities / intensities.max()) * 100
                            fragments = []
                            idx = 0
                            pep_len = len(peptide)
                            # generate+1 fragments
                            for ion_type in ['b', 'y']:
                                for ion_num in range(1, pep_len):
                                    for loss in [None, 'H2O', 'NH3']:
                                        if idx < len(intensities):
                                            mz = calculate_fragment_mz(peptide, ion_type, ion_num, 1, loss)
                                            intensity = intensities[idx]
                                            label = f"{ion_type}{ion_num}"
                                            if loss:
                                                label += f"-{loss}"
                                                fragments.append({
                                                    'mz': mz,
                                                    'intensity': intensity,
                                                    'label': label,
                                                    'ion_type': ion_type,
                                                    'ion_num': ion_num,
                                                    'charge': 1
                                                })
                                                idx += 1
                                                # optional: +2
                                                if include_charge2:
                                                    for ion_type in ['b', 'y']:
                                                        for ion_num in range(1, pep_len):
                                                            for loss in [None]:
                                                                base_label = f"{ion_type}{ion_num}"
                                                                base_peaks = [f for f in fragments if f['label'] == base_label and f['charge'] == 1]
                                                                if base_peaks and base_peaks[0]['intensity'] > 5:
                                                                    mz = calculate_fragment_mz(peptide, ion_type, ion_num, 2, loss)
                                                                    intensity = base_peaks[0]['intensity'] * 0.5
                                                                    label = f"{ion_type}{ion_num}++"
                                                                    fragments.append({
                                                                        'mz': mz,
                                                                        'intensity': intensity,
                                                                        'label': label,
                                                                        'ion_type': ion_type,
                                                                        'ion_num': ion_num,
                                                                        'charge': 2
                                                                    })
                                                                    print(f" OK prediction succeeded, to {len(fragments)} fragments")
                                                                    return pd.DataFrame(fragments)
        except Exception as e:
            print(f" Failed error: {e}")
            return None

def annotate_experimental_peaks(exp_df, peptide, charge=2, mz_tolerance=0.02):
    """forexperimental spectra annotation"""
    if exp_df is None or len(exp_df) == 0:
        return exp_df
    exp_df = exp_df.copy()
    exp_df['label'] = ''
    exp_df['ion_type'] = ''
    exp_df['charge'] = 1
    pep_len = len(peptide)
    for idx, peak in exp_df.iterrows():
        peak_mz = peak['mz']
        best_match = None
        best_error = mz_tolerance
        for frag_charge in [1, 2]:
            for ion_type in ['b', 'y']:
                for ion_num in range(1, pep_len):
                    for loss in [None, 'H2O', 'NH3']:
                        theo_mz = calculate_fragment_mz(peptide, ion_type, ion_num, frag_charge, loss)
                        error = abs(peak_mz - theo_mz)
                        if error < best_error:
                            best_error = error
                            label = f"{ion_type}{ion_num}"
                            if frag_charge > 1:
                                label += "+" * frag_charge
                                if loss:
                                    label += f"-{loss}"
                                    best_match = (label, ion_type, frag_charge)
                                    if best_match:
                                        exp_df.at[idx, 'label'] = best_match[0]
                                        exp_df.at[idx, 'ion_type'] = best_match[1]
                                        exp_df.at[idx, 'charge'] = best_match[2]
                                        return exp_df

def save_spectrum_data(peptide, charge, exp_df, pred_df, output_file, sample_name="", peptide_info=None, model_name="", sa_score=None, n_matched=0):
"""
save dataforCSVformat, R
"""
# data
metadata = {
    'peptide': peptide,
    'charge': charge,
    'sample': sample_name,
    'model': model_name,
    'sa_score': sa_score if sa_score is not None else 'NA',
    'n_matched_peaks': n_matched,
    'n_exp_peaks': len(exp_df) if exp_df is not None else 0,
    'n_pred_peaks': len(pred_df) if pred_df is not None else 0
}
if peptide_info:
    for key in ['intensity', 'rt', 'mz', 'n_replicates']:
        if key in peptide_info:
            metadata[key] = peptide_info[key]
            # save data
            metadata_file = output_file.replace('.csv', '_metadata.json')
            with open(metadata_file, 'w') as f:
                json.dump(metadata, f, indent=2)
                # experimental spectradata
                if exp_df is not None and len(exp_df) > 0:
                    exp_df_save = exp_df.copy()
                    if exp_df_save['intensity'].max() > 0:
                        exp_df_save['intensity'] = (exp_df_save['intensity'] / exp_df_save['intensity'].max()) * 100
                        exp_df_save = annotate_experimental_peaks(exp_df_save, peptide, charge)
                        exp_df_save['spectrum'] = 'experimental'
                    else:
                        exp_df_save = pd.DataFrame(columns=['mz', 'intensity', 'label', 'ion_type', 'charge', 'spectrum'])
                        # data
                        if pred_df is not None and len(pred_df) > 0:
                            pred_df_save = pred_df[pred_df['intensity'] > 1].copy()
                            pred_df_save['spectrum'] = 'predicted'
                        else:
                            pred_df_save = pd.DataFrame(columns=['mz', 'intensity', 'label', 'ion_type', 'charge', 'spectrum'])
                            # data
                            combined_df = pd.concat([exp_df_save, pred_df_save], ignore_index=True)
                            # saveforCSV
                            combined_df.to_csv(output_file, index=False)
                            print(f" OK save data: {Path(output_file).name}")
                            print(f" Summary: {model_name}")
                            if sa_score is not None:
                                print(f" SA = {sa_score:.3f} ({n_matched} matched )")
                            else:
                                print(f" Warning no calculateSA (matched )")

def select_best_psm(df_sample, intensity_col=None):
    """from PSMin """
    n_psms = len(df_sample)
    if intensity_col is None:
        for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
            if col in df_sample.columns:
                intensity_col = col
                break
            if intensity_col and intensity_col in df_sample.columns:
                best_idx = df_sample[intensity_col].idxmax()
                best_row = df_sample.loc[best_idx]
                avg_intensity = df_sample[intensity_col].mean()
                std_intensity = df_sample[intensity_col].std()
            else:
                best_row = df_sample.iloc[0]
                avg_intensity = None
                std_intensity = None
                stats = {
                    'n_psms': n_psms,
                    'avg_intensity': avg_intensity,
                    'std_intensity': std_intensity,
                    'intensity_col': intensity_col
                }
                return best_row, stats

def main():
    """ """
    print("="*70)
    print("DIA-NN data ( )")
    print("="*70)
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"\noutput directory: {output_path}")
    # Processing section
    print(f"\nSummary:")
    for model_key in MODELS_TO_TEST:
        if model_key in AVAILABLE_MODELS:
            model_config = AVAILABLE_MODELS[model_key]
            print(f" - {model_key}:")
            print(f" Summary: {model_config['model_name']}")
            if model_config.get('requires_instrument'):
                print(f" Summary: {model_config.get('default_instrument', 'N/A')}")
                print(f" NCE: {model_config.get('default_nce', 30)}")
                # readDIA-NN
                lib_tsv_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p-lib.tsv"
                exp_spectra = {}
                if lib_tsv_file.exists():
                    exp_spectra = read_diann_lib_tsv(lib_tsv_file)
                else:
                    print(f"\nWarning not foundDIA-NN file: {lib_tsv_file.name}")
                    print(f" onlysave ")
                    # readDIA-NN report
                    print(f"\n{'='*70}")
                    print("readDIA-NN results")
                    print("="*70)
                    diann_file = Path(DIANN_DIR) / "20251212_FXC_MiTRI_HLA1p.tsv"
                    if not diann_file.exists():
                        print(f"Failed error: file does not exist {diann_file}")
                        return
                    print(f"readfile: {diann_file.name}")
                    df = pd.read_csv(diann_file, sep='\t', low_memory=False)
                    print(f"OK load {len(df)} rows of data")
                    # identified key columns
                    peptide_col = None
                    for col in ['Stripped.Sequence', 'Modified.Sequence', 'Peptide']:
                        if col in df.columns:
                            peptide_col = col
                            break
                        if not peptide_col:
                            print("Failed error: not foundpeptide sequencecolumn")
                            return
                        run_col = None
                        for col in ['Run', 'File.Name', 'R.FileName']:
                            if col in df.columns:
                                run_col = col
                                break
                            charge_col = None
                            for col in ['Precursor.Charge', 'Charge']:
                                if col in df.columns:
                                    charge_col = col
                                    break
                                rt_col = 'RT' if 'RT' in df.columns else None
                                mz_col = 'Precursor.Mz' if 'Precursor.Mz' in df.columns else None
                                intensity_col = None
                                for col in ['PG.Quantity', 'Precursor.Quantity', 'MS1.Area', 'Intensity']:
                                    if col in df.columns:
                                        intensity_col = col
                                        break
                                    # filtertargetpeptide
                                    print(f"\n{'='*70}")
                                    print("processtargetpeptide")
                                    print("="*70)
                                    df_filtered = df[df[peptide_col].isin(PEPTIDES)].copy()
                                    print(f"OK found {len(df_filtered)} matched record")
                                    if len(df_filtered) == 0:
                                        print("\nWarning warning: not foundtargetpeptide")
                                        return
                                    if run_col:
                                        samples = sorted(df_filtered[run_col].unique())
                                        print(f" {len(samples)} samples")
                                    else:
                                        samples = [None]
                                        # compare results
                                        model_comparison = []
                                        total_saved = 0
                                        for peptide in PEPTIDES:
                                            print(f"\n{'-'*70}")
                                            print(f"peptide: {peptide}")
                                            print("-"*70)
                                            df_peptide = df_filtered[df_filtered[peptide_col] == peptide]
                                            if len(df_peptide) == 0:
                                                continue
                                            for sample in samples:
                                                if sample and run_col:
                                                    df_sample = df_peptide[df_peptide[run_col] == sample]
                                                    sample_short = Path(sample).stem if sample else "Unknown"
                                                else:
                                                    df_sample = df_peptide
                                                    sample_short = "All"
                                                    if len(df_sample) == 0:
                                                        continue
                                                    best_row, stats = select_best_psm(df_sample, intensity_col)
                                                    charge = int(best_row[charge_col]) if charge_col else 2
                                                    peptide_info = {}
                                                    if rt_col and rt_col in best_row:
                                                        peptide_info['rt'] = best_row[rt_col]
                                                        if mz_col and mz_col in best_row:
                                                            peptide_info['mz'] = best_row[mz_col]
                                                            if intensity_col and intensity_col in best_row:
                                                                peptide_info['intensity'] = best_row[intensity_col]
                                                                peptide_info['n_replicates'] = stats['n_psms']
                                                                print(f"\n sample: {sample_short}")
                                                                print(f" Summary: {charge}")
                                                                print(f" PSMs: {stats['n_psms']}")
                                                                # experimental spectra
                                                                exp_key = f"{peptide}_z{charge}"
                                                                exp_df = exp_spectra.get(exp_key, None)
                                                                if exp_df is not None:
                                                                    n_exp_peaks = len(exp_df)
                                                                    print(f" experimental spectra: {n_exp_peaks} ")
                                                                    if n_exp_peaks < 20:
                                                                        print(f" Warning, SA ")
                                                                    else:
                                                                        print(f" Warning not foundexperimental spectra")
                                                                        # each
                                                                        for model_key in MODELS_TO_TEST:
                                                                            if model_key not in AVAILABLE_MODELS:
                                                                                print(f" Failed unknown model: {model_key}")
                                                                                continue
                                                                            model_config = AVAILABLE_MODELS[model_key]
                                                                            print(f"\n Summary: {model_key}")
                                                                            # Processing section
                                                                            pred_df = get_prediction(
                                                                                peptide, charge=charge, model_config=model_config,
                                                                                collision_energy=model_config.get('default_nce', 30),
                                                                                include_charge2=True
)
# calculateSA
sa_score = None
n_matched = 0
if pred_df is not None and exp_df is not None:
    sa_score, n_matched = calculate_spectral_angle(exp_df, pred_df)
    # recordcompareresults
    model_comparison.append({
        'peptide': peptide,
        'sample': sample_short,
        'charge': charge,
        'model': model_key,
        'sa_score': sa_score,
        'n_matched': n_matched,
        'n_exp_peaks': len(exp_df),
        'n_pred_peaks': len(pred_df)
    })
    # save data
    output_file = output_path / f"{sample_short}_{peptide}_z{charge}_{model_key}_data.csv"
    save_spectrum_data(
        peptide, charge, exp_df, pred_df, str(output_file),
        sample_name=sample_short,
        peptide_info=peptide_info,
        model_name=model_key,
        sa_score=sa_score,
        n_matched=n_matched
)
total_saved += 1
# save compare results
if model_comparison:
    comparison_df = pd.DataFrame(model_comparison)
    comparison_file = output_path / "model_comparison.csv"
    comparison_df.to_csv(comparison_file, index=False)
    print(f"\n{'='*70}")
    print(" compare results:")
    print("="*70)
    # peptide
    for peptide in comparison_df['peptide'].unique():
        print(f"\npeptide: {peptide}")
        peptide_data = comparison_df[comparison_df['peptide'] == peptide]
        for model in peptide_data['model'].unique():
            model_data = peptide_data[peptide_data['model'] == model]
            avg_sa = model_data['sa_score'].dropna().mean()
            avg_matched = model_data['n_matched'].mean()
            if not np.isnan(avg_sa):
                print(f" {model}: averageSA = {avg_sa:.3f}, average matched = {avg_matched:.1f}")
            else:
                print(f" {model}: Warning nowith SA ")
                print(f"\n{'='*70}")
                print(f"OK completed!saved total {total_saved} data files")
                print(f"output directory: {output_path}")
                if model_comparison:
                    print(f" compare results: {output_path}/model_comparison.csv")
                    print("="*70)
                    # Processing section
                    if model_comparison:
                        low_sa_cases = [x for x in model_comparison if x['sa_score'] is not None and x['sa_score'] < 0.5]
                        if low_sa_cases:
                            print(f"\nWarning Summary:")
                            print(f" {len(low_sa_cases)} SA (<0.5) ")
                            print(f"Possible reason:")
                            print(f" 1. DIA-NN in (<20)")
                            print(f" 2. rowsNCE ( NCE 20-40 )")
                            print(f" 3. experimental spectra ")
                            print(f"\nSummary:")
                            print(f" - checkDIA-NN ")
                            print(f" - NCE ")
                            print(f" - AlphaPeptDeep, instrument correct")

if __name__ == "__main__":
    main()